# Kappa Foundation — Investment Policy and Initial Allocation

**Case decision date:** 31 December 2022  
**Estimation window:** January 2015–December 2022; the effective common return sample is determined programmatically after aligning Yahoo and Fama–French observations  
**Frozen evaluation window:** January 2023–December 2025

This notebook is organized around the investment recommendation rather than lecture order. It funds the two mandatory nominal payments first, then evaluates a diversified U.S. equity growth sleeve for the 2030 real-value objective. All initial portfolio inputs are restricted to information available by 31 December 2022. The 2023–2025 data are used only for the frozen out-of-sample evaluation.

The notebook is designed to rebuild from **saved raw data without internet access**. The canonical Yahoo rebuild input is the saved raw CSV (portable across pandas versions); the raw pickle is retained only as an archival convenience copy. Kenneth French factors are reparsed directly from the saved raw text. A separate retrieval cell is included to refresh those raw files when internet access is available. Because the historical files were retrieved after the 2022 decision date, revised-history and retrospective security-selection limitations are disclosed rather than treated as a genuinely unseen live experiment.

**Planning assumptions:** 2.5% inflation is treated as a conservative long-run planning input rather than a forecast and is stress-tested from 2% to 4%. The 10 bp one-way trading-cost allowance is a simple implementation assumption for liquid U.S.-listed equities/ETFs, not an executable quote; the final memo should cite or otherwise justify both assumptions using information available by the 2022 cutoff.


### Portfolio objective and constraints

The growth portfolio is selected using mean–variance utility rather than an unconstrained maximum-Sharpe rule. This allows the trustees’ risk preference to enter the decision explicitly through the risk-aversion parameter. The optimisation is subject to the case restrictions of no leverage and no short selling, together with position limits to avoid an excessively concentrated solution.


In [ ]:
# ============================================================
# 01A. SETUP, CASE FACTS AND MODELLING ASSUMPTIONS
# ============================================================

import numpy as np
import pandas as pd
import statsmodels.api as sm
import matplotlib.pyplot as plt
import sys, platform

from scipy.optimize import brentq, minimize
from scipy.stats import norm
from pathlib import Path
from IPython.display import display

pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

# Works whether the notebook is run from /notebooks or project root.
CWD = Path.cwd()
PROJECT_ROOT = CWD.parent if CWD.name.lower() == "notebooks" else CWD
RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
TABLE_DIR = PROJECT_ROOT / "outputs" / "tables"
FIGURE_DIR = PROJECT_ROOT / "outputs" / "figures"

for directory in [RAW_DIR, PROCESSED_DIR, TABLE_DIR, FIGURE_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

# -------------------------
# Case facts
# -------------------------
DECISION_DATE = pd.Timestamp("2022-12-31")
INITIAL_CAPITAL = 12_000_000
LIABILITIES = {2024: 2_400_000, 2027: 3_600_000}
REAL_TARGET_2030 = 8_000_000
FACE_VALUE = 1_000

BOND_DATA = pd.DataFrame({
    "Bond": ["Alpha", "Beta", "Gamma", "Delta", "Epsilon"],
    "Maturity": [1, 2, 3, 5, 8],
    "CouponRate": [0.000, 0.000, 0.040, 0.035, 0.045],
    "Price": [958.77, 922.78, 1002.49, 986.07, 1040.71],
})

# -------------------------
# Information windows
# -------------------------
ESTIMATION_START = "2015-01-01"
ESTIMATION_END = "2022-12-31"
EVALUATION_START = "2023-01-01"
EVALUATION_END = "2025-12-31"

# -------------------------
# Team assumptions / policy choices
# -------------------------
# Inflation is a planning assumption, not a promised forecast.
BASE_INFLATION = 0.025

# A=4 is a moderate base preference, not inferred mechanically from the 15% concern.
# The required comparison is performed for A = 2, 4, 6, 8.
BASE_RISK_AVERSION = 4
RISK_AVERSION_VALUES = [2, 4, 6, 8]

# Additional concentration limits chosen by the team.
MAX_STOCK_WEIGHT = 0.15
MAX_ETF_WEIGHT = 0.35

# One-way implementation cost assumption.
BASE_TRADING_COST = 0.001  # 10 bps

# Bond stress magnitudes. These are stress scenarios, not forecasts.
PARALLEL_SHOCK = 0.0100      # +/-100 bp
CURVE_SHAPE_SHOCK = 0.0075   # +/-75 bp slope change

NOMINAL_TARGET_2030 = REAL_TARGET_2030 * (1 + BASE_INFLATION) ** 8


Table_A2_Environment = pd.DataFrame({
    "Component": ["Python", "pandas", "numpy", "statsmodels"],
    "Version": [
        platform.python_version(),
        pd.__version__,
        np.__version__,
        sm.__version__,
    ],
})
Table_A2_Environment.to_csv(TABLE_DIR / "Table_A2_Environment.csv", index=False)

Table_A1_Mandate_Assumptions = pd.DataFrame({
    "Item": [
        "Initial capital",
        "2024 mandatory payment",
        "2027 mandatory payment",
        "2030 real-value objective",
        "Base inflation assumption",
        "2030 nominal target",
        "Trustee loss concern",
        "Base risk aversion A",
        "Risk-aversion comparison",
        "Maximum stock weight",
        "Maximum ETF weight",
        "One-way equity/ETF trading cost",
        "Bond trading-cost treatment",
    ],
    "Value": [
        INITIAL_CAPITAL,
        LIABILITIES[2024],
        LIABILITIES[2027],
        REAL_TARGET_2030,
        BASE_INFLATION,
        NOMINAL_TARGET_2030,
        "Losses approaching 15%; horizon/metric unspecified",
        BASE_RISK_AVERSION,
        str(RISK_AVERSION_VALUES),
        MAX_STOCK_WEIGHT,
        MAX_ETF_WEIGHT,
        BASE_TRADING_COST,
        "No additional cost beyond case executable quote",
    ],
    "Treatment": [
        "Case fact", "Case fact", "Case fact", "Case fact",
        "Team planning assumption; document source/rationale in final memo",
        "Derived from real target and inflation assumption",
        "Case fact; treated as diagnostic rather than hard constraint",
        "Base preference parameter; sensitivity required",
        "Shows how assumed risk preference changes the choice",
        "Additional concentration limit",
        "Additional concentration limit",
        "Team implementation assumption for liquid listed growth assets",
        "Case quotes are stated to be executable; no extra bond trading spread/cost is added",
    ],
})

display(Table_A1_Mandate_Assumptions)


# 2. Bond Valuation and Funding Risk

The liability-funding sleeve is designed to reduce the risk that equity-market conditions interfere with the foundation’s scheduled payments.

The 2024 obligation is matched with the two-year zero-coupon bond. The 2027 obligation is funded with the five-year Delta bond, with all interim coupons retained for the payment date under a conservative 0% reinvestment assumption.

Because the case provides only a partial term structure, the spot curve is bootstrapped from the available bond prices. Duration and convexity are then compared with exact repricing under both parallel and non-parallel yield-curve shocks. The purpose of this analysis is not to forecast bond returns, but to test the sensitivity of the funding plan to interest-rate changes while preserving the hold-to-maturity payment structure.


### Why this step is needed

The case provides bond prices and contractual cash flows, but not a complete spot curve. A spot curve is therefore bootstrapped from the available instruments so that each bond can be valued using cash-flow-specific discount rates. The resulting curve is used only for valuation and interest-rate sensitivity analysis; it is not treated as a forecast of future rates.


In [ ]:
# ============================================================
# 02A. BOND FUNCTIONS AND ONE EXPLICIT TERM-STRUCTURE METHOD
# ============================================================

def bond_cashflows(face, coupon_rate, maturity):
    coupon = face * coupon_rate
    cashflows = np.full(int(maturity), coupon, dtype=float)
    cashflows[-1] += face
    return cashflows


def price_from_ytm(face, coupon_rate, maturity, ytm):
    cashflows = bond_cashflows(face, coupon_rate, maturity)
    t = np.arange(1, int(maturity) + 1)
    return np.sum(cashflows / (1 + ytm) ** t)


def solve_ytm(face, coupon_rate, maturity, market_price):
    objective = lambda y: price_from_ytm(face, coupon_rate, maturity, y) - market_price
    return brentq(objective, -0.50, 1.00)


def duration_convexity(face, coupon_rate, maturity, ytm):
    cashflows = bond_cashflows(face, coupon_rate, maturity)
    t = np.arange(1, int(maturity) + 1)
    pv = cashflows / (1 + ytm) ** t
    price = pv.sum()

    macaulay = np.sum(t * pv) / price
    modified = macaulay / (1 + ytm)
    convexity = (
        np.sum(cashflows * t * (t + 1) / (1 + ytm) ** (t + 2)) / price
    )
    return macaulay, modified, convexity


def price_from_spot_curve(face, coupon_rate, maturity, spot_curve):
    cashflows = bond_cashflows(face, coupon_rate, maturity)
    return sum(
        cashflows[t - 1] / (1 + spot_curve[t]) ** t
        for t in range(1, int(maturity) + 1)
    )


def linear_spot(t, t_left, r_left, t_right, r_right):
    weight = (t - t_left) / (t_right - t_left)
    return r_left + weight * (r_right - r_left)


def build_spot_curve():
    """Bootstrap 1Y, 2Y, 3Y, 5Y, 8Y; linearly interpolate 4Y, 6Y, 7Y."""
    curve = {}

    alpha = BOND_DATA.loc[BOND_DATA["Bond"] == "Alpha"].iloc[0]
    beta = BOND_DATA.loc[BOND_DATA["Bond"] == "Beta"].iloc[0]
    gamma = BOND_DATA.loc[BOND_DATA["Bond"] == "Gamma"].iloc[0]
    delta = BOND_DATA.loc[BOND_DATA["Bond"] == "Delta"].iloc[0]
    epsilon = BOND_DATA.loc[BOND_DATA["Bond"] == "Epsilon"].iloc[0]

    curve[1] = FACE_VALUE / alpha["Price"] - 1
    curve[2] = (FACE_VALUE / beta["Price"]) ** (1 / 2) - 1

    gamma_coupon = FACE_VALUE * gamma["CouponRate"]
    remaining_gamma_cf = (
        gamma["Price"]
        - gamma_coupon / (1 + curve[1])
        - gamma_coupon / (1 + curve[2]) ** 2
    )
    curve[3] = ((FACE_VALUE + gamma_coupon) / remaining_gamma_cf) ** (1 / 3) - 1

    def delta_error(s5):
        s4 = linear_spot(4, 3, curve[3], 5, s5)
        temp = {1: curve[1], 2: curve[2], 3: curve[3], 4: s4, 5: s5}
        return price_from_spot_curve(
            FACE_VALUE, delta["CouponRate"], int(delta["Maturity"]), temp
        ) - delta["Price"]

    curve[5] = brentq(delta_error, -0.25, 0.50)
    curve[4] = linear_spot(4, 3, curve[3], 5, curve[5])

    def epsilon_error(s8):
        s6 = linear_spot(6, 5, curve[5], 8, s8)
        s7 = linear_spot(7, 5, curve[5], 8, s8)
        temp = {
            1: curve[1], 2: curve[2], 3: curve[3], 4: curve[4], 5: curve[5],
            6: s6, 7: s7, 8: s8,
        }
        return price_from_spot_curve(
            FACE_VALUE, epsilon["CouponRate"], int(epsilon["Maturity"]), temp
        ) - epsilon["Price"]

    curve[8] = brentq(epsilon_error, -0.25, 0.50)
    curve[6] = linear_spot(6, 5, curve[5], 8, curve[8])
    curve[7] = linear_spot(7, 5, curve[5], 8, curve[8])
    return curve


spot_curve = build_spot_curve()
Table_B1_Term_Structure = pd.DataFrame({
    "MaturityYear": sorted(spot_curve),
    "SpotRate": [spot_curve[t] for t in sorted(spot_curve)],
    "Status": [
        "Bootstrapped" if t in {1, 2, 3, 5, 8} else "Linear spot interpolation"
        for t in sorted(spot_curve)
    ],
})

display(Table_B1_Term_Structure)

### Duration, convexity and exact repricing

Duration and convexity provide a compact approximation of price sensitivity to yield changes, but the assignment requires those estimates to be checked against exact repricing. The code therefore compares the approximation with a full cash-flow repricing under the same rate shock. This shows both the direction and the approximation error of the risk measures.


In [ ]:
# ============================================================
# 02B. YTM, DURATION/CONVEXITY AND EXACT REPRICING CHECK
# ============================================================

DC_CHECK_SHOCK = 0.0100  # +/-100 bp local YTM check
bond_rows = []
dc_rows = []

for _, row in BOND_DATA.iterrows():
    ytm = solve_ytm(FACE_VALUE, row["CouponRate"], int(row["Maturity"]), row["Price"])
    macaulay, modified, convexity = duration_convexity(
        FACE_VALUE, row["CouponRate"], int(row["Maturity"]), ytm
    )
    spot_reprice = price_from_spot_curve(
        FACE_VALUE, row["CouponRate"], int(row["Maturity"]), spot_curve
    )

    bond_rows.append({
        "Bond": row["Bond"],
        "MarketPrice": row["Price"],
        "YTM": ytm,
        "MacaulayDuration": macaulay,
        "ModifiedDuration": modified,
        "Convexity": convexity,
        "SpotCurveReprice": spot_reprice,
        "SpotRepricingError": spot_reprice - row["Price"],
    })

    for shock in [-DC_CHECK_SHOCK, DC_CHECK_SHOCK]:
        exact_price = price_from_ytm(
            FACE_VALUE, row["CouponRate"], int(row["Maturity"]), ytm + shock
        )
        exact_change = exact_price / row["Price"] - 1
        approx_change = -modified * shock + 0.5 * convexity * shock ** 2
        dc_rows.append({
            "Bond": row["Bond"],
            "YieldShock": shock,
            "ExactPriceChangePct": exact_change,
            "DurationConvexityApproxPct": approx_change,
            "ApproximationErrorPctPts": approx_change - exact_change,
        })

Table_B2_Bond_Analytics = pd.DataFrame(bond_rows)
Table_B3_Duration_Convexity_Check = pd.DataFrame(dc_rows)

# Internal QA: the chosen curve should reproduce quoted bonds closely.
if Table_B2_Bond_Analytics["SpotRepricingError"].abs().max() > 1e-4:
    raise ValueError("Spot-curve construction does not reproduce quoted bond prices closely enough.")

display(Table_B2_Bond_Analytics)
display(Table_B3_Duration_Convexity_Check)

### Liability-funding logic

The two mandatory payments are separated from the growth objective. The two-year zero-coupon bond is used for the 2024 payment because its maturity date matches the obligation directly. The five-year Delta bond is used for the 2027 payment, and all interim coupons are retained for that obligation under a 0% reinvestment assumption. This is deliberately conservative because the funding plan does not depend on earning an additional return on the coupons.


In [ ]:
# ============================================================
# 02C. FUNDED-PAYMENT PLAN
# ============================================================

beta = BOND_DATA.loc[BOND_DATA["Bond"] == "Beta"].iloc[0]
delta = BOND_DATA.loc[BOND_DATA["Bond"] == "Delta"].iloc[0]

# 2024: exact 2-year zero-coupon maturity match.
beta_units = LIABILITIES[2024] / FACE_VALUE
beta_initial_cost = beta_units * beta["Price"]
beta_contractual_cash_2024 = beta_units * FACE_VALUE

# 2027: Delta matures in year 5. Interim coupons are ring-fenced as cash.
# Base funding assumption: 0% reinvestment on pre-2027 coupons.
delta_cashflows_per_bond = bond_cashflows(
    FACE_VALUE, delta["CouponRate"], int(delta["Maturity"])
)
delta_2027_value_per_bond_at_zero_reinvestment = delta_cashflows_per_bond.sum()
delta_units = LIABILITIES[2027] / delta_2027_value_per_bond_at_zero_reinvestment
delta_initial_cost = delta_units * delta["Price"]
delta_contractual_cash_2027 = delta_units * delta_2027_value_per_bond_at_zero_reinvestment

LIABILITY_BUCKET_COST = beta_initial_cost + delta_initial_cost
GROWTH_BUDGET_BEFORE_COSTS = INITIAL_CAPITAL - LIABILITY_BUCKET_COST

Table_B4_Funding_Plan = pd.DataFrame({
    "Liability": ["2024 scholarship", "2027 learning centre"],
    "RequiredPayment": [LIABILITIES[2024], LIABILITIES[2027]],
    "FundingAsset": ["Beta", "Delta"],
    "Units": [beta_units, delta_units],
    "InitialCost": [beta_initial_cost, delta_initial_cost],
    "ContractualCashAvailable": [beta_contractual_cash_2024, delta_contractual_cash_2027],
    "FundingRatio": [
        beta_contractual_cash_2024 / LIABILITIES[2024],
        delta_contractual_cash_2027 / LIABILITIES[2027],
    ],
    "FundingMethod": [
        "2Y zero-coupon maturity match",
        "5Y Delta; interim coupons ring-fenced; 0% base reinvestment",
    ],
})

display(Table_B4_Funding_Plan)
print(f"Up-front liability reserve cost: USD {LIABILITY_BUCKET_COST:,.2f}")
print(f"Residual budget available for the growth sleeve before trading cost: USD {GROWTH_BUDGET_BEFORE_COSTS:,.2f}")

In [ ]:
# DYNAMIC FUNDING INTERPRETATION
min_funding_ratio = Table_B4_Funding_Plan["FundingRatio"].min()
if min_funding_ratio >= 1.0 - 1e-10:
    print(
        f"Funding conclusion: both mandatory payments are fully covered by contractual bond cash flows "
        f"(minimum funding ratio = {min_funding_ratio:.3f}). The growth sleeve therefore does not need "
        "to be liquidated to meet the 2024 or 2027 commitments under the stated hold-to-maturity assumptions."
    )
else:
    print(
        f"Funding warning: the minimum funding ratio is {min_funding_ratio:.3f}; "
        "the liability sleeve does not fully cover all mandatory payments."
    )


### Duration, convexity and exact repricing

Duration and convexity provide a compact approximation of price sensitivity to yield changes, but the assignment requires those estimates to be checked against exact repricing. The code therefore compares the approximation with a full cash-flow repricing under the same rate shock. This shows both the direction and the approximation error of the risk measures.


In [ ]:
# ============================================================
# 02D. REQUIRED RATE STRESSES AND LINK TO PAYMENT OBLIGATIONS
# ============================================================

def shift_spot_curve(base_curve, parallel_shift=0.0, slope_shift=0.0):
    maturities = sorted(base_curve)
    midpoint = (min(maturities) + max(maturities)) / 2
    half_range = (max(maturities) - min(maturities)) / 2

    shocked = {}
    for t in maturities:
        slope_loading = (t - midpoint) / half_range
        shocked[t] = base_curve[t] + parallel_shift + slope_shift * slope_loading
    return shocked


# +/-100 bp is a transparent material parallel shock; +/-75 bp across the curve
# is used as a distinct non-parallel shape stress rather than as a forecast.
curve_scenarios = {
    "Base": (0.0, 0.0),
    "Parallel +100bp": (PARALLEL_SHOCK, 0.0),
    "Parallel -100bp": (-PARALLEL_SHOCK, 0.0),
    "Steepener (+/-75bp slope)": (0.0, CURVE_SHAPE_SHOCK),
    "Flattener (+/-75bp slope)": (0.0, -CURVE_SHAPE_SHOCK),
}

price_rows = []
liability_rows = []

for scenario, (parallel, slope) in curve_scenarios.items():
    shocked_curve = shift_spot_curve(spot_curve, parallel, slope)

    scenario_prices = {}
    for _, row in BOND_DATA.iterrows():
        p = price_from_spot_curve(
            FACE_VALUE, row["CouponRate"], int(row["Maturity"]), shocked_curve
        )
        scenario_prices[row["Bond"]] = p
        price_rows.append({
            "Scenario": scenario,
            "Bond": row["Bond"],
            "ExactRepricedPrice": p,
            "PriceChangePct": p / row["Price"] - 1,
        })

    liability_rows.append({
        "Scenario": scenario,
        "BetaMarketValue": beta_units * scenario_prices["Beta"],
        "DeltaMarketValue": delta_units * scenario_prices["Delta"],
        "TotalLiabilitySleeveMarketValue": (
            beta_units * scenario_prices["Beta"] + delta_units * scenario_prices["Delta"]
        ),
        "2024ContractualFundingRatioIfHeld": 1.0,
        "2027ContractualFundingRatioIfHeldAndCouponsRingFenced": 1.0,
    })

Table_B6_Rate_Scenarios = pd.DataFrame(price_rows)
Table_B7_Liability_Sleeve_Under_Rate_Stress = pd.DataFrame(liability_rows)

# Exact repricing changes mark-to-market values. The held-to-maturity contractual
# payment cash flows remain unchanged under these default-free case assumptions.
display(Table_B6_Rate_Scenarios)
display(Table_B7_Liability_Sleeve_Under_Rate_Stress)

for name, table in {
    "Table_A1_Mandate_Assumptions": Table_A1_Mandate_Assumptions,
    "Table_B1_Term_Structure": Table_B1_Term_Structure,
    "Table_B2_Bond_Analytics": Table_B2_Bond_Analytics,
    "Table_B3_Duration_Convexity_Check": Table_B3_Duration_Convexity_Check,
    "Table_B4_Funding_Plan": Table_B4_Funding_Plan,
    "Table_B6_Rate_Scenarios": Table_B6_Rate_Scenarios,
    "Table_B7_Liability_Sleeve_Under_Rate_Stress": Table_B7_Liability_Sleeve_Under_Rate_Stress,
}.items():
    table.to_csv(TABLE_DIR / f"{name}.csv", index=False)


### Term-structure assumption sensitivity

The quoted bonds do not identify a unique annual spot curve because there are fewer traded maturities than annual cash-flow dates. The base case therefore uses linear interpolation in spot rates. To test whether the bond-risk conclusions depend materially on that convention, the same quoted prices are also fitted using log-linear interpolation in discount factors. Both curves exactly reprice the observed bonds; differences at the unobserved maturities measure interpolation-model uncertainty rather than market mispricing.


In [ ]:
# ============================================================
# 02E. TERM-STRUCTURE INTERPOLATION SENSITIVITY
# ============================================================

def discount_factor_from_spot(rate, maturity):
    return 1.0 / (1.0 + rate) ** maturity

def spot_from_discount_factor(df, maturity):
    return df ** (-1.0 / maturity) - 1.0

def loglinear_df(t, t_left, df_left, t_right, df_right):
    weight = (t - t_left) / (t_right - t_left)
    return np.exp(
        np.log(df_left) + weight * (np.log(df_right) - np.log(df_left))
    )

def build_spot_curve_loglinear_discount():
    """
    Alternative feasible curve:
    - exact 1Y, 2Y and 3Y discount factors are bootstrapped;
    - 4Y is log-linearly interpolated in discount factors between 3Y and 5Y;
    - 6Y and 7Y are log-linearly interpolated between 5Y and 8Y;
    - 5Y and 8Y are solved so Delta and Epsilon exactly reprice.
    """
    alpha = BOND_DATA.loc[BOND_DATA["Bond"] == "Alpha"].iloc[0]
    beta = BOND_DATA.loc[BOND_DATA["Bond"] == "Beta"].iloc[0]
    gamma = BOND_DATA.loc[BOND_DATA["Bond"] == "Gamma"].iloc[0]
    delta = BOND_DATA.loc[BOND_DATA["Bond"] == "Delta"].iloc[0]
    epsilon = BOND_DATA.loc[BOND_DATA["Bond"] == "Epsilon"].iloc[0]

    df = {}
    df[1] = alpha["Price"] / FACE_VALUE
    df[2] = beta["Price"] / FACE_VALUE

    gamma_coupon = FACE_VALUE * gamma["CouponRate"]
    df[3] = (
        gamma["Price"]
        - gamma_coupon * df[1]
        - gamma_coupon * df[2]
    ) / (FACE_VALUE + gamma_coupon)

    def delta_error(log_df5):
        df5 = np.exp(log_df5)
        df4 = loglinear_df(4, 3, df[3], 5, df5)
        coupon = FACE_VALUE * delta["CouponRate"]
        price = (
            coupon * df[1] + coupon * df[2] + coupon * df[3]
            + coupon * df4 + (FACE_VALUE + coupon) * df5
        )
        return price - delta["Price"]

    log_df5 = brentq(delta_error, np.log(0.40), np.log(1.50))
    df[5] = np.exp(log_df5)
    df[4] = loglinear_df(4, 3, df[3], 5, df[5])

    def epsilon_error(log_df8):
        df8 = np.exp(log_df8)
        df6 = loglinear_df(6, 5, df[5], 8, df8)
        df7 = loglinear_df(7, 5, df[5], 8, df8)
        coupon = FACE_VALUE * epsilon["CouponRate"]
        price = (
            coupon * df[1] + coupon * df[2] + coupon * df[3]
            + coupon * df[4] + coupon * df[5]
            + coupon * df6 + coupon * df7
            + (FACE_VALUE + coupon) * df8
        )
        return price - epsilon["Price"]

    log_df8 = brentq(epsilon_error, np.log(0.25), np.log(1.80))
    df[8] = np.exp(log_df8)
    df[6] = loglinear_df(6, 5, df[5], 8, df[8])
    df[7] = loglinear_df(7, 5, df[5], 8, df[8])

    return {t: spot_from_discount_factor(df[t], t) for t in sorted(df)}

spot_curve_alt = build_spot_curve_loglinear_discount()

# Verify that both feasible curves exactly reproduce every observed quote.
curve_repricing_rows = []
for curve_name, curve in {
    "Linear spot interpolation": spot_curve,
    "Log-linear discount-factor interpolation": spot_curve_alt,
}.items():
    for _, row in BOND_DATA.iterrows():
        repriced = price_from_spot_curve(
            FACE_VALUE, row["CouponRate"], int(row["Maturity"]), curve
        )
        curve_repricing_rows.append({
            "CurveMethod": curve_name,
            "Bond": row["Bond"],
            "ObservedPrice": row["Price"],
            "RepricedPrice": repriced,
            "AbsoluteError": abs(repriced - row["Price"]),
        })

Table_B8_Curve_Repricing_Check = pd.DataFrame(curve_repricing_rows)
if Table_B8_Curve_Repricing_Check["AbsoluteError"].max() > 1e-6:
    raise ValueError("A term-structure sensitivity curve does not exactly reprice the observed bonds.")

Table_B9_Interpolation_Sensitivity = pd.DataFrame({
    "MaturityYear": sorted(spot_curve),
    "LinearSpotRate": [spot_curve[t] for t in sorted(spot_curve)],
    "LogLinearDiscountSpotRate": [spot_curve_alt[t] for t in sorted(spot_curve)],
})
Table_B9_Interpolation_Sensitivity["Difference_bp"] = (
    Table_B9_Interpolation_Sensitivity["LogLinearDiscountSpotRate"]
    - Table_B9_Interpolation_Sensitivity["LinearSpotRate"]
) * 10_000

# Re-run the same rate stresses under the alternative curve and compare liability-sleeve market values.
interp_stress_rows = []
for curve_name, curve in {
    "Linear spot interpolation": spot_curve,
    "Log-linear discount-factor interpolation": spot_curve_alt,
}.items():
    for scenario, (parallel, slope) in curve_scenarios.items():
        shocked_curve = shift_spot_curve(curve, parallel, slope)
        beta_price = price_from_spot_curve(
            FACE_VALUE, beta["CouponRate"], int(beta["Maturity"]), shocked_curve
        )
        delta_price = price_from_spot_curve(
            FACE_VALUE, delta["CouponRate"], int(delta["Maturity"]), shocked_curve
        )
        interp_stress_rows.append({
            "CurveMethod": curve_name,
            "Scenario": scenario,
            "LiabilitySleeveMarketValue": beta_units * beta_price + delta_units * delta_price,
            "2024ContractualFundingRatioIfHeld": 1.0,
            "2027ContractualFundingRatioIfHeldAndCouponsRingFenced": 1.0,
        })

Table_B10_Interpolation_Stress_Sensitivity = pd.DataFrame(interp_stress_rows)

display(Table_B8_Curve_Repricing_Check)
display(Table_B9_Interpolation_Sensitivity)
display(Table_B10_Interpolation_Stress_Sensitivity)

for name, table in {
    "Table_B8_Curve_Repricing_Check": Table_B8_Curve_Repricing_Check,
    "Table_B9_Interpolation_Sensitivity": Table_B9_Interpolation_Sensitivity,
    "Table_B10_Interpolation_Stress_Sensitivity": Table_B10_Interpolation_Stress_Sensitivity,
}.items():
    table.to_csv(TABLE_DIR / f"{name}.csv", index=False)


In [ ]:
# DYNAMIC TERM-STRUCTURE SENSITIVITY INTERPRETATION
max_spot_difference_bp = Table_B9_Interpolation_Sensitivity["Difference_bp"].abs().max()

stress_pivot = Table_B10_Interpolation_Stress_Sensitivity.pivot(
    index="Scenario", columns="CurveMethod", values="LiabilitySleeveMarketValue"
)
stress_method_difference = (
    stress_pivot["Log-linear discount-factor interpolation"]
    - stress_pivot["Linear spot interpolation"]
).abs().max()

print(
    f"Interpolation sensitivity: the largest difference between the two feasible spot curves "
    f"is {max_spot_difference_bp:.2f} bp at the unobserved maturities, and the largest difference "
    f"in stressed liability-sleeve market value is USD {stress_method_difference:,.0f}. "
    "Contractual funding ratios remain unchanged at 1.0 under hold-to-maturity because the "
    "interpolation method affects mark-to-market valuation, not the promised cash flows."
)


# 3. Data, Estimation Window and Reproducibility

All market information used for the initial decision is restricted to data available by 31 December 2022.

Monthly market data are obtained from Yahoo Finance and Fama–French factor data from Kenneth French’s Data Library. The effective common estimation sample is determined after aligning the Yahoo and Fama–French monthly observations. January 2023 to December 2025 is reserved exclusively for out-of-sample evaluation.

The notebook is designed to rebuild the analysis from saved raw files without manual edits or internet access. Source records, download settings, package versions and security-eligibility evidence are retained separately for reproducibility.


In [ ]:
# ============================================================
# 03A. FIXED CANDIDATE UNIVERSE AND ELIGIBILITY EVIDENCE
# ============================================================

STOCKS = [
    "AAPL", "MSFT", "JPM", "JNJ", "PG",
    "HD", "CAT", "CVX", "KO", "WMT",
]
ETFS = ["SPY", "IWM"]
BENCHMARK = "SPY"
CANDIDATE_TICKERS = STOCKS + ETFS

sector_or_mandate = {
    "AAPL": "Information Technology",
    "MSFT": "Information Technology",
    "JPM": "Financials",
    "JNJ": "Health Care",
    "PG": "Consumer Staples",
    "HD": "Consumer Discretionary",
    "CAT": "Industrials",
    "CVX": "Energy",
    "KO": "Consumer Staples",
    "WMT": "Consumer Staples",
    "SPY": "Broad U.S. large-cap equity ETF",
    "IWM": "U.S. small-cap equity ETF",
}

# Explicit universe QA against the assignment.
if len(STOCKS) < 8 or len(STOCKS) > 12:
    raise ValueError("Assignment requires 8–12 U.S. common stocks.")
if len(ETFS) != 2:
    raise ValueError("Assignment requires exactly two U.S.-listed U.S.-equity ETFs.")
stock_sectors = {sector_or_mandate[t] for t in STOCKS}
if len(stock_sectors) < 4:
    raise ValueError("Assignment requires common stocks across at least four sectors.")
if sector_or_mandate[ETFS[0]] == sector_or_mandate[ETFS[1]]:
    raise ValueError("The two ETFs must have meaningfully different mandates.")

Table_D1_Candidate_Universe = pd.DataFrame({
    "Ticker": CANDIDATE_TICKERS,
    "SecurityType": ["ETF" if t in ETFS else "Common stock" for t in CANDIDATE_TICKERS],
    "SectorOrMandate": [sector_or_mandate[t] for t in CANDIDATE_TICKERS],
    "Role": [
        "Simple investable benchmark and market proxy" if t == BENCHMARK
        else "Growth/diversification candidate"
        for t in CANDIDATE_TICKERS
    ],
})

# Required evidence file — do not substitute present-day survival or price existence.
ELIGIBILITY_FILE = RAW_DIR / "security_eligibility_2022.csv"
required_eligibility_columns = {
    "Ticker",
    "SecurityTypeAtDecisionDate",
    "SectorOrMandateAtDecisionDate",
    "EligibleAtDecisionDate",
    "EvidenceSource",
    "EvidenceDateOrArchiveDate",
    "SelectionRationale",
}

if not ELIGIBILITY_FILE.exists():
    raise FileNotFoundError(
        "Missing data/raw/security_eligibility_2022.csv. "
        "Create dated case-date eligibility evidence before final submission."
    )

Table_D2_Eligibility_Evidence = pd.read_csv(ELIGIBILITY_FILE)
missing_columns = required_eligibility_columns - set(Table_D2_Eligibility_Evidence.columns)
if missing_columns:
    raise ValueError(f"Eligibility evidence is missing columns: {sorted(missing_columns)}")

eligibility = Table_D2_Eligibility_Evidence.set_index("Ticker").reindex(CANDIDATE_TICKERS)
if eligibility.isna().all(axis=1).any():
    missing = eligibility.index[eligibility.isna().all(axis=1)].tolist()
    raise ValueError(f"Missing case-date eligibility evidence for: {missing}")

eligible_flag = eligibility["EligibleAtDecisionDate"].astype(str).str.lower().isin(
    ["true", "1", "yes", "y"]
)
if not eligible_flag.all():
    failed = eligibility.index[~eligible_flag].tolist()
    raise ValueError(f"These candidates are not verified as eligible at the case date: {failed}")

# Evidence itself must be contemporaneous with (or earlier than) the case decision date.
evidence_dates = pd.to_datetime(eligibility["EvidenceDateOrArchiveDate"], errors="coerce")
if evidence_dates.isna().any():
    raise ValueError("Eligibility evidence contains an invalid evidence/archive date.")
if (evidence_dates > DECISION_DATE).any():
    failed = evidence_dates.index[evidence_dates > DECISION_DATE].tolist()
    raise ValueError(f"Post-decision eligibility evidence detected for: {failed}")
if eligibility["EvidenceSource"].astype(str).str.strip().eq("").any():
    raise ValueError("Eligibility evidence contains a blank source.")


# Validate that the evidence describes the correct security type and case-date role,
# rather than merely carrying a pre-filled eligibility flag.
expected_security_type = {
    t: ("ETF" if t in ETFS else "Common stock")
    for t in CANDIDATE_TICKERS
}
for ticker in CANDIDATE_TICKERS:
    actual_type = str(eligibility.loc[ticker, "SecurityTypeAtDecisionDate"]).strip().lower()
    expected_type = expected_security_type[ticker].lower()
    if expected_type not in actual_type and actual_type not in expected_type:
        raise ValueError(
            f"Eligibility evidence has an inconsistent security type for {ticker}: "
            f"{eligibility.loc[ticker, 'SecurityTypeAtDecisionDate']!r}"
        )

    stated_role = str(eligibility.loc[ticker, "SectorOrMandateAtDecisionDate"]).strip()
    if not stated_role:
        raise ValueError(f"Missing case-date sector/mandate evidence for {ticker}.")

    selection_reason = str(eligibility.loc[ticker, "SelectionRationale"]).strip()
    if not selection_reason:
        raise ValueError(f"Missing selection rationale for {ticker}.")

# The evidence sources must point to dated archival/filing material rather than present-day pages.
source_text = eligibility["EvidenceSource"].astype(str).str.lower()
if not source_text.str.contains("sec.gov").all():
    raise ValueError("All eligibility records must be supported by dated SEC filing/archive sources.")

display(Table_D1_Candidate_Universe)
display(Table_D2_Eligibility_Evidence)


### Market-data construction

The decision is restricted to information available by 31 December 2022. Raw Yahoo Finance data are retained separately from processed returns so that the analysis can be rebuilt without manual intervention. Monthly total returns are constructed consistently across all candidate securities, and the 2023–2025 period is excluded from estimation and reserved for evaluation.


In [ ]:
# ============================================================
# 03B. FUNCTIONING DATA-RETRIEVAL CODE — DISABLED BY DEFAULT
# ============================================================
# The assignment requires both saved raw data for offline rebuilding and
# functioning retrieval code. Rebuild the frozen analysis from the submitted
# saved raw files with RUN_DATA_RETRIEVAL = False.

RUN_DATA_RETRIEVAL = False

if RUN_DATA_RETRIEVAL:
    import io
    import re
    import zipfile
    import urllib.request
    from datetime import datetime, timezone
    import yfinance as yf

    retrieval_timestamp = datetime.now(timezone.utc).isoformat()

    # Match the frozen saved-data design: raw prices start Jan-2015, so the
    # first computable common return is Feb-2015.
    YAHOO_REQUEST_START = "2015-01-01"
    YAHOO_REQUEST_END_EXCLUSIVE = "2026-01-01"

    market_raw_download = yf.download(
        tickers=CANDIDATE_TICKERS,
        start=YAHOO_REQUEST_START,
        end=YAHOO_REQUEST_END_EXCLUSIVE,
        interval="1d",
        auto_adjust=False,
        actions=True,
        group_by="column",
        threads=True,
        progress=False,
    )
    if market_raw_download.empty:
        raise RuntimeError("Yahoo Finance returned an empty download.")

    if isinstance(market_raw_download.columns, pd.MultiIndex):
        lvl0 = set(map(str, market_raw_download.columns.get_level_values(0)))
        if not ({"Adj Close", "Close"} & lvl0):
            market_raw_download = market_raw_download.swaplevel(0, 1, axis=1).sort_index(axis=1)
    elif len(CANDIDATE_TICKERS) != 1:
        raise ValueError("Unexpected non-MultiIndex Yahoo response for multiple tickers.")

    # Keep the original downloaded response in two convenient representations.
    market_raw_download.to_pickle(RAW_DIR / "yahoo_market_data_raw.pkl")
    market_raw_download.to_csv(RAW_DIR / "yahoo_market_data_raw.csv")

    adj_close_download = market_raw_download["Adj Close"][CANDIDATE_TICKERS]
    yahoo_diag = pd.DataFrame({
        "Ticker": CANDIDATE_TICKERS,
        "NonMissingAdjClose": [int(adj_close_download[t].notna().sum()) for t in CANDIDATE_TICKERS],
        "FirstValidDate": [adj_close_download[t].first_valid_index() for t in CANDIDATE_TICKERS],
        "LastValidDate": [adj_close_download[t].last_valid_index() for t in CANDIDATE_TICKERS],
    })
    yahoo_diag.to_csv(RAW_DIR / "yahoo_download_diagnostics.csv", index=False)

    pd.DataFrame([{
        "Source": "Yahoo Finance via yfinance",
        "Dataset": "Daily market price history",
        "RetrievalDate": retrieval_timestamp,
        "Tickers": ",".join(CANDIDATE_TICKERS),
        "RequestedStartDate": YAHOO_REQUEST_START,
        "RequestedEndDateExclusive": YAHOO_REQUEST_END_EXCLUSIVE,
        "AutoAdjust": False,
        "Actions": True,
        "GroupBy": "column",
        "Threads": True,
        "ReturnPriceField": "Adj Close",
        "CorporateActionTreatment": "Monthly returns are constructed from Yahoo Adjusted Close",
        "RawPickleFile": "yahoo_market_data_raw.pkl",
        "RawCSVFile": "yahoo_market_data_raw.csv",
        "DiagnosticFile": "yahoo_download_diagnostics.csv",
        "YFinanceVersion": yf.__version__,
        "PandasVersion": pd.__version__,
        "PythonVersion": platform.python_version(),
    }]).to_csv(RAW_DIR / "yahoo_source_record.csv", index=False)

    # Kenneth French: preserve original ZIP plus extracted raw text.
    FRENCH_URL = (
        "https://mba.tuck.dartmouth.edu/pages/faculty/ken.french/ftp/"
        "F-F_Research_Data_Factors_CSV.zip"
    )
    french_zip_bytes = urllib.request.urlopen(FRENCH_URL, timeout=60).read()
    (RAW_DIR / "F-F_Research_Data_Factors_CSV.zip").write_bytes(french_zip_bytes)

    with zipfile.ZipFile(io.BytesIO(french_zip_bytes)) as zf:
        csv_name = next(name for name in zf.namelist() if name.lower().endswith(".csv"))
        raw_text = zf.read(csv_name).decode("utf-8", errors="replace")
    (RAW_DIR / "F-F_Research_Data_Factors_raw.txt").write_text(raw_text, encoding="utf-8")

    rows = []
    for line in raw_text.splitlines():
        parts = [p.strip() for p in line.split(",")]
        if parts and re.fullmatch(r"\d{6}", parts[0]) and len(parts) >= 5:
            rows.append(parts[:5])
    if not rows:
        raise ValueError("Could not parse monthly Kenneth French factor rows.")

    french = pd.DataFrame(rows, columns=["YYYYMM", "Mkt-RF", "SMB", "HML", "RF"])
    french["Date"] = pd.to_datetime(french["YYYYMM"], format="%Y%m") + pd.offsets.MonthEnd(0)
    for col in ["Mkt-RF", "SMB", "HML", "RF"]:
        french[col] = pd.to_numeric(french[col], errors="raise") / 100.0
    french = french.set_index("Date")[["Mkt-RF", "SMB", "HML", "RF"]].sort_index()

    french_full = french.loc["2015-01-31":"2025-12-31"].copy()
    french_est = french_full.loc["2015-01-31":"2022-12-31"].copy()
    french_eval = french_full.loc["2023-01-31":"2025-12-31"].copy()
    french_full.to_csv(PROCESSED_DIR / "french_factors_monthly_2015_2025.csv", index_label="Date")
    french_est.to_csv(PROCESSED_DIR / "french_factors_estimation_2015_2022.csv", index_label="Date")
    french_eval.to_csv(PROCESSED_DIR / "french_factors_evaluation_2023_2025.csv", index_label="Date")

    pd.DataFrame([{
        "Source": "Kenneth French Data Library",
        "DatasetIdentifier": "F-F Research Data Factors",
        "SourceURL": FRENCH_URL,
        "RetrievalDate": retrieval_timestamp,
        "RawZipFile": "F-F_Research_Data_Factors_CSV.zip",
        "RawTextFile": "F-F_Research_Data_Factors_raw.txt",
        "RawUnits": "Percent per month",
        "ProcessedUnits": "Decimal return per month",
        "DecisionInformationCutoff": "2022-12-31",
        "EstimationWindow": "2015-01 to 2022-12 factors; common return sample begins 2015-02",
        "EvaluationWindow": "2023-01 to 2025-12",
        "HistoricalRevisionCaveat": "Retrieved after the 2022 decision date; revised historical data may differ from contemporaneous files",
        "PythonPackage": "urllib / pandas",
        "PandasVersion": pd.__version__,
        "PythonVersion": platform.python_version(),
    }]).to_csv(RAW_DIR / "french_source_record.csv", index=False)

    print("Raw Yahoo response, French ZIP/raw text, processed factor files and source records saved.")
else:
    print("Data retrieval skipped. Offline analysis will use saved raw files.")


### Market-data construction

The decision is restricted to information available by 31 December 2022. Raw Yahoo Finance data are retained separately from processed returns so that the analysis can be rebuilt without manual intervention. Monthly total returns are constructed consistently across all candidate securities, and the 2023–2025 period is excluded from estimation and reserved for evaluation.


In [ ]:
# ============================================================
# 03C. LOAD SAVED RAW YAHOO DATA AND CONSTRUCT MONTHLY RETURNS
# ============================================================

RAW_MARKET_CSV_FILE = RAW_DIR / "yahoo_market_data_raw.csv"
RAW_MARKET_PICKLE_FILE = RAW_DIR / "yahoo_market_data_raw.pkl"
YAHOO_SOURCE_RECORD_FILE = RAW_DIR / "yahoo_source_record.csv"

# Canonical offline rebuild source: CSV. Pickles can be incompatible across pandas
# versions, so the submitted pickle is archival/convenience evidence only.
for required_file in [RAW_MARKET_CSV_FILE, YAHOO_SOURCE_RECORD_FILE]:
    if not required_file.exists():
        raise FileNotFoundError(
            f"Missing {required_file}. Run the market-data retrieval script first."
        )

market_raw = pd.read_csv(
    RAW_MARKET_CSV_FILE,
    header=[0, 1],
    index_col=0,
    parse_dates=True,
)
Table_D3_Yahoo_Source_Record = pd.read_csv(YAHOO_SOURCE_RECORD_FILE)

market_raw.index = pd.to_datetime(market_raw.index)
market_raw = market_raw.sort_index()
if market_raw.index.has_duplicates:
    raise ValueError("Duplicate Yahoo dates detected in saved raw data.")

if not isinstance(market_raw.columns, pd.MultiIndex):
    raise ValueError("Expected saved Yahoo data to have MultiIndex columns.")

available_fields = market_raw.columns.get_level_values(0).unique().tolist()
available_tickers = market_raw.columns.get_level_values(1).unique().tolist()

missing_tickers = [t for t in CANDIDATE_TICKERS if t not in available_tickers]
if missing_tickers:
    raise ValueError(f"Missing candidate tickers in Yahoo raw data: {missing_tickers}")
if "Adj Close" not in available_fields:
    raise ValueError("Adj Close is missing; do not continue return construction.")

adjusted_close_daily = market_raw["Adj Close"][CANDIDATE_TICKERS].copy()
monthly_prices = adjusted_close_daily.resample("ME").last()
monthly_returns = monthly_prices.pct_change(fill_method=None)

estimation_precheck = monthly_returns.loc[
    "2015-01-31":"2022-12-31", CANDIDATE_TICKERS
].copy()
evaluation_precheck = monthly_returns.loc[
    "2023-01-31":"2025-12-31", CANDIDATE_TICKERS
].copy()

# No imputation: use common complete months only.
estimation_returns = estimation_precheck.dropna(how="any")
evaluation_returns = evaluation_precheck.dropna(how="any")

if len(estimation_returns) < 60:
    raise ValueError("Fewer than 60 common monthly observations in 2015–2022.")
if estimation_returns.index.max() > DECISION_DATE:
    raise ValueError("Post-2022 information leaked into the estimation sample.")
if evaluation_returns.index.min() < pd.Timestamp("2023-01-31") or evaluation_returns.index.max() > pd.Timestamp("2025-12-31"):
    raise ValueError("Evaluation returns fall outside the required 2023-2025 window.")

# The first return in the estimation window may be mechanically unavailable because
# pct_change requires a prior-month price. Separate that from genuine missing observations.
initial_return_unavailable = estimation_precheck.iloc[0].isna()
other_estimation_missing = estimation_precheck.iloc[1:].isna().sum()
evaluation_missing = evaluation_precheck.isna().sum()

Table_D4_Return_Construction = pd.DataFrame({
    "Item": [
        "Market-data source",
        "Raw frequency",
        "Monthly sampling",
        "Return definition",
        "Corporate-action treatment",
        "Estimation window",
        "Evaluation window",
        "Missing-data treatment",
        "Benchmark",
        "Retrospective-selection limitation",
    ],
    "Treatment": [
        "Yahoo Finance via yfinance; saved raw response",
        "Daily",
        "Last available adjusted close in each calendar month",
        "P_adj,t / P_adj,t-1 - 1",
        "Yahoo Adjusted Close; retrieval settings documented in source record",
        "Requested Jan 2015-Dec 2022; effective common returns Feb 2015-Dec 2022 (95 months)",
        "Jan 2023-Dec 2025; isolated from initial decision",
        "Common complete months only; no imputation",
        BENCHMARK,
        "Case universe selected retrospectively; disclosed rather than presented as an unbiased live sample",
    ],
})

monthly_prices.to_csv(PROCESSED_DIR / "monthly_adjusted_prices.csv")
monthly_returns.to_csv(PROCESSED_DIR / "monthly_returns_full.csv")
estimation_returns.to_csv(PROCESSED_DIR / "monthly_returns_estimation_2015_2022.csv")
evaluation_returns.to_csv(PROCESSED_DIR / "monthly_returns_evaluation_2023_2025.csv")

print(f"Estimation observations: {len(estimation_returns)}")
print(f"Evaluation observations: {len(evaluation_returns)}")

if initial_return_unavailable.all() and other_estimation_missing.sum() == 0 and evaluation_missing.sum() == 0:
    print(
        "The only unavailable estimation return is the first month, which is mechanical: "
        "a prior-month price is required to compute a return. No other asset-month observations are missing."
    )
else:
    missing_summary = pd.DataFrame({
        "Ticker": CANDIDATE_TICKERS,
        "InitialReturnUnavailable": initial_return_unavailable.values,
        "OtherEstimationMissing": other_estimation_missing.values,
        "EvaluationMissing": evaluation_missing.values,
    })
    print("Missing observations detected; no values are imputed:")
    display(missing_summary)

display(Table_D3_Yahoo_Source_Record)
display(Table_D4_Return_Construction)

for name, table in {
    "Table_D1_Candidate_Universe": Table_D1_Candidate_Universe,
    "Table_D2_Eligibility_Evidence": Table_D2_Eligibility_Evidence,
    "Table_D3_Yahoo_Source_Record": Table_D3_Yahoo_Source_Record,
    "Table_D4_Return_Construction": Table_D4_Return_Construction,
}.items():
    table.to_csv(TABLE_DIR / f"{name}.csv", index=False)


# 4. Growth Portfolio Construction and Recommendation

The candidate universe contains ten US common stocks across multiple sectors and two US-listed equity ETFs with distinct mandates. SPY is used as the investable comparator for the growth allocation.

Expected returns are estimated using a CAPM-style approach with betas estimated from the pre-2023 sample and the market risk premium estimated from the same historical window. Portfolio risk is evaluated under both the sample covariance matrix and a single-index covariance model.

The recommended allocation is selected using mean–variance utility subject to no short selling, no leverage and position limits. Risk-aversion sensitivity is reported to show how the preferred portfolio changes as the trustees place greater weight on downside risk.

A historical maximum-Sharpe portfolio is retained as a credible rejected alternative. It provides a useful comparison because it represents the treasurer’s preference for the strongest historical risk-adjusted performance, but it is more exposed to estimation error and concentration risk.


### Factor-data construction

Fama–French monthly factors are used in two places: the market excess return supports the CAPM-style expected-return inputs, and the FF3 model is used later to explain the economic exposures of the growth portfolio. Factor observations are aligned by month with the market-return data so that all regressions use a common sample.


In [ ]:
# ============================================================
# 04A. ALIGN PRE-2023 FRENCH FACTORS AND BUILD EXPECTED RETURNS
# ============================================================

FRENCH_RAW_TEXT_FILE = RAW_DIR / "F-F_Research_Data_Factors_raw.txt"
FRENCH_SOURCE_RECORD_FILE = RAW_DIR / "french_source_record.csv"
FRENCH_FULL_FILE = PROCESSED_DIR / "french_factors_monthly_2015_2025.csv"
FRENCH_ESTIMATION_FILE = PROCESSED_DIR / "french_factors_estimation_2015_2022.csv"
FRENCH_EVALUATION_FILE = PROCESSED_DIR / "french_factors_evaluation_2023_2025.csv"

for required_file in [FRENCH_RAW_TEXT_FILE, FRENCH_SOURCE_RECORD_FILE]:
    if not required_file.exists():
        raise FileNotFoundError(
            f"Missing {required_file}. The offline rebuild must start from saved raw French data."
        )

# Reparse the saved Kenneth French raw text on every clean run. This avoids a hidden
# dependence on pre-created processed factor files and satisfies raw-to-analysis rebuildability.
import re
raw_french_text = FRENCH_RAW_TEXT_FILE.read_text(encoding="utf-8", errors="replace")
rows = []
for line in raw_french_text.splitlines():
    parts = [p.strip() for p in line.split(",")]
    if parts and re.fullmatch(r"\d{6}", parts[0]) and len(parts) >= 5:
        rows.append(parts[:5])
if not rows:
    raise ValueError("Could not parse monthly Kenneth French factor rows from saved raw text.")

french_all = pd.DataFrame(rows, columns=["YYYYMM", "Mkt-RF", "SMB", "HML", "RF"])
french_all["Date"] = pd.to_datetime(french_all["YYYYMM"], format="%Y%m") + pd.offsets.MonthEnd(0)
for col in ["Mkt-RF", "SMB", "HML", "RF"]:
    french_all[col] = pd.to_numeric(french_all[col], errors="raise") / 100.0
french_all = french_all.set_index("Date")[["Mkt-RF", "SMB", "HML", "RF"]].sort_index()


# Unit QA: Kenneth French raw values are percentages per month and are converted above
# to decimal monthly returns. These bounds are deliberately broad but catch a missed /100 conversion.
if french_all[["Mkt-RF", "SMB", "HML"]].abs().max().max() >= 0.50:
    raise ValueError("French factor magnitudes are inconsistent with decimal monthly units.")
if french_all["RF"].abs().max() >= 0.02:
    raise ValueError("French RF magnitudes are inconsistent with decimal monthly units.")

french_full = french_all.loc["2015-01-31":"2025-12-31"].copy()
french_estimation = french_full.loc["2015-01-31":"2022-12-31"].copy()
french_evaluation = french_full.loc["2023-01-31":"2025-12-31"].copy()

if len(french_estimation) != 96 or len(french_evaluation) != 36:
    raise ValueError("Unexpected French factor sample size for the required case windows.")

french_full.to_csv(FRENCH_FULL_FILE, index_label="Date")
french_estimation.to_csv(FRENCH_ESTIMATION_FILE, index_label="Date")
french_evaluation.to_csv(FRENCH_EVALUATION_FILE, index_label="Date")
Table_P1_French_Source_Record = pd.read_csv(FRENCH_SOURCE_RECORD_FILE)

common_dates = estimation_returns.index.intersection(french_estimation.index)
returns_for_model = estimation_returns.loc[common_dates, CANDIDATE_TICKERS].copy()
factors_for_model = french_estimation.loc[common_dates].copy()

if len(common_dates) < 60:
    raise ValueError("Fewer than 60 common Yahoo/French monthly observations.")
if common_dates.max() > DECISION_DATE:
    raise ValueError("Post-2022 factor information leaked into estimation.")

rf_monthly = factors_for_model["RF"]
market_excess_monthly = factors_for_model["Mkt-RF"]
BASE_RF_ANNUAL = rf_monthly.mean() * 12
BASE_MARKET_PREMIUM_ANNUAL = market_excess_monthly.mean() * 12

expected_return_rows = []
for ticker in CANDIDATE_TICKERS:
    asset_return = returns_for_model[ticker]
    asset_excess = asset_return - rf_monthly
    X = sm.add_constant(market_excess_monthly)
    model = sm.OLS(asset_excess, X).fit()

    beta = model.params["Mkt-RF"]
    expected_return_rows.append({
        "Ticker": ticker,
        "CAPMBeta": beta,
        "CAPMAlphaMonthly_DiagnosticOnly": model.params["const"],
        "BaseExpectedReturn": BASE_RF_ANNUAL + beta * BASE_MARKET_PREMIUM_ANNUAL,
        "HistoricalAnnualMean_SensitivityOnly": asset_return.mean() * 12,
    })

Table_P2_Expected_Return_Inputs = pd.DataFrame(expected_return_rows).set_index("Ticker")
BASE_EXPECTED_RETURNS = Table_P2_Expected_Return_Inputs["BaseExpectedReturn"]
HISTORICAL_EXPECTED_RETURNS = Table_P2_Expected_Return_Inputs[
    "HistoricalAnnualMean_SensitivityOnly"
]

print("Base expected returns use RF + beta × market risk premium. Estimated CAPM alpha is reported only as a diagnostic and is not added to the forecast.")
display(Table_P1_French_Source_Record)
display(Table_P2_Expected_Return_Inputs)


### Why two covariance models are compared

The assignment requires the portfolio decision to be tested under both the sample covariance matrix and a single-index risk model. The sample covariance matrix preserves the full historical cross-sectional dependence, while the single-index model imposes a simpler structure in which co-movement is largely explained by market beta. Running the same optimisation under both risk estimates shows how dependent the recommended weights are on the covariance specification.


In [ ]:
# ============================================================
# 04B. REQUIRED SAMPLE-COVARIANCE AND SINGLE-INDEX RISK MODELS
# ============================================================

candidate_returns = returns_for_model[CANDIDATE_TICKERS].copy()
SAMPLE_COV = candidate_returns.cov() * 12

market_returns = returns_for_model[BENCHMARK]
market_variance_monthly = market_returns.var(ddof=1)

si_betas = {}
si_residual_variances = {}
for ticker in CANDIDATE_TICKERS:
    if ticker == BENCHMARK:
        si_betas[ticker] = 1.0
        si_residual_variances[ticker] = 0.0
    else:
        X = sm.add_constant(market_returns)
        model = sm.OLS(candidate_returns[ticker], X).fit()
        si_betas[ticker] = model.params[BENCHMARK]
        si_residual_variances[ticker] = model.resid.var(ddof=1)

single_index_monthly = pd.DataFrame(
    index=CANDIDATE_TICKERS, columns=CANDIDATE_TICKERS, dtype=float
)
for i in CANDIDATE_TICKERS:
    for j in CANDIDATE_TICKERS:
        systematic = si_betas[i] * si_betas[j] * market_variance_monthly
        single_index_monthly.loc[i, j] = systematic + (
            si_residual_variances[i] if i == j else 0.0
        )

SINGLE_INDEX_COV = single_index_monthly * 12

# Internal numerical QA only; not a material decision output.
if np.linalg.eigvalsh(SAMPLE_COV.values).min() < -1e-10:
    raise ValueError("Sample covariance is not positive semidefinite within tolerance.")
if np.linalg.eigvalsh(SINGLE_INDEX_COV.values).min() < -1e-10:
    raise ValueError("Single-index covariance is not positive semidefinite within tolerance.")

### Portfolio objective and constraints

The growth portfolio is selected using mean–variance utility rather than an unconstrained maximum-Sharpe rule. This allows the trustees’ risk preference to enter the decision explicitly through the risk-aversion parameter. The optimisation is subject to the case restrictions of no leverage and no short selling, together with position limits to avoid an excessively concentrated solution.


In [ ]:
# ============================================================
# 04C. PORTFOLIO FUNCTIONS, CONSTRAINTS AND BASE DECISIONS
# ============================================================

BASE_BOUNDS = [
    (0.0, MAX_ETF_WEIGHT if t in ETFS else MAX_STOCK_WEIGHT)
    for t in CANDIDATE_TICKERS
]


def portfolio_statistics(weights, expected_returns, covariance):
    mu = np.asarray(expected_returns)
    sigma = np.asarray(covariance)
    expected_return = weights @ mu
    variance = weights @ sigma @ weights
    return expected_return, np.sqrt(variance), variance


def optimize_mean_variance_utility(expected_returns, covariance, risk_aversion, bounds):
    mu = np.asarray(expected_returns)
    sigma = np.asarray(covariance)
    n = len(mu)

    def objective(w):
        return -(w @ mu - 0.5 * risk_aversion * (w @ sigma @ w))

    result = minimize(
        objective,
        np.repeat(1 / n, n),
        method="SLSQP",
        bounds=bounds,
        constraints=[{"type": "eq", "fun": lambda w: w.sum() - 1}],
        options={"maxiter": 1000, "ftol": 1e-12},
    )
    if not result.success:
        raise RuntimeError(f"Utility optimization failed: {result.message}")
    return result.x


def optimize_max_sharpe(expected_returns, covariance, risk_free_rate, bounds):
    mu = np.asarray(expected_returns)
    sigma = np.asarray(covariance)
    n = len(mu)

    def objective(w):
        vol = np.sqrt(w @ sigma @ w)
        if vol <= 0:
            return 1e6
        return -((w @ mu - risk_free_rate) / vol)

    result = minimize(
        objective,
        np.repeat(1 / n, n),
        method="SLSQP",
        bounds=bounds,
        constraints=[{"type": "eq", "fun": lambda w: w.sum() - 1}],
        options={"maxiter": 1000, "ftol": 1e-12},
    )
    if not result.success:
        raise RuntimeError(f"Max-Sharpe optimization failed: {result.message}")
    return result.x


Table_P3_Constraints = pd.DataFrame({
    "Constraint": [
        "Long only",
        "Fully invested growth sleeve",
        "Maximum common-stock weight",
        "Maximum ETF weight",
        "Borrowing",
        "Short selling",
        "Derivatives",
    ],
    "Policy": [
        "Yes", "Yes", MAX_STOCK_WEIGHT, MAX_ETF_WEIGHT,
        "Prohibited", "Prohibited", "Prohibited",
    ],
    "Reason": [
        "Mandate",
        "Compare complete growth allocations on the same capital base",
        "Limit single-name concentration and estimation error",
        "Permit a diversified ETF core without allowing one ETF to dominate",
        "Mandate", "Mandate", "Mandate",
    ],
})

display(Table_P3_Constraints)

In [ ]:
# ============================================================
# 04D. REQUIRED RISK-MODEL COMPARISON
# ============================================================

mu_base = BASE_EXPECTED_RETURNS.loc[CANDIDATE_TICKERS].values

sample_weights = optimize_mean_variance_utility(
    mu_base, SAMPLE_COV.values, BASE_RISK_AVERSION, BASE_BOUNDS
)
single_index_weights = optimize_mean_variance_utility(
    mu_base, SINGLE_INDEX_COV.values, BASE_RISK_AVERSION, BASE_BOUNDS
)

Table_P4_Risk_Model_Comparison = pd.DataFrame({
    "Ticker": CANDIDATE_TICKERS,
    "SampleCovWeight": sample_weights,
    "SingleIndexWeight": single_index_weights,
    "Difference": single_index_weights - sample_weights,
})

risk_model_turnover = 0.5 * np.abs(single_index_weights - sample_weights).sum()

summary_rows = []
for name, weights, cov in [
    ("Sample covariance", sample_weights, SAMPLE_COV.values),
    ("Single-index covariance", single_index_weights, SINGLE_INDEX_COV.values),
]:
    er, vol, var = portfolio_statistics(weights, mu_base, cov)
    summary_rows.append({
        "RiskModel": name,
        "ExpectedReturn": er,
        "Volatility": vol,
        "Utility_A4_UnderOwnRiskModel": er - 0.5 * BASE_RISK_AVERSION * var,
        "LargestWeight": weights.max(),
    })

Table_P5_Risk_Model_Summary = pd.DataFrame(summary_rows)
Table_P5_Risk_Model_Summary["AllocationTurnoverVsOtherModel"] = risk_model_turnover

display(Table_P4_Risk_Model_Comparison)
display(Table_P5_Risk_Model_Summary)
print("Each row above is evaluated under the same covariance model used to optimize that row. A common comparison lens is applied later when choosing the initial policy.")

In [ ]:
# ============================================================
# 04E. RISK-AVERSION COMPARISON + ONE EXPECTED-RETURN SENSITIVITY
# ============================================================

risk_aversion_rows = []
for model_name, covariance in {
    "Sample covariance": SAMPLE_COV.values,
    "Single-index covariance": SINGLE_INDEX_COV.values,
}.items():
    for A in RISK_AVERSION_VALUES:
        w = optimize_mean_variance_utility(mu_base, covariance, A, BASE_BOUNDS)
        er, vol, var = portfolio_statistics(w, mu_base, covariance)
        top3_idx = np.argsort(w)[::-1][:3]
        top3 = ", ".join(f"{CANDIDATE_TICKERS[i]} {w[i]:.1%}" for i in top3_idx)
        risk_aversion_rows.append({
            "RiskModel": model_name,
            "RiskAversionA": A,
            "ExpectedReturn": er,
            "Volatility": vol,
            "Utility": er - 0.5 * A * var,
            "LargestWeight": w.max(),
            "Top3Holdings": top3,
        })

Table_P6_Risk_Aversion_Sensitivity = pd.DataFrame(risk_aversion_rows)

# Compact sensitivity to the expected-return specification.
mu_historical = HISTORICAL_EXPECTED_RETURNS.loc[CANDIDATE_TICKERS].values
historical_mu_weights = optimize_mean_variance_utility(
    mu_historical, SINGLE_INDEX_COV.values, BASE_RISK_AVERSION, BASE_BOUNDS
)
expected_return_turnover = 0.5 * np.abs(historical_mu_weights - single_index_weights).sum()

base_er, base_vol, base_var = portfolio_statistics(
    single_index_weights, mu_base, SINGLE_INDEX_COV.values
)
hist_er_under_hist_mu, hist_vol, hist_var = portfolio_statistics(
    historical_mu_weights, mu_historical, SINGLE_INDEX_COV.values
)

Table_P7_Expected_Return_Sensitivity = pd.DataFrame({
    "ExpectedReturnSpecification": ["CAPM-style base", "Historical arithmetic mean"],
    "ExpectedReturn": [base_er, hist_er_under_hist_mu],
    "Volatility": [base_vol, hist_vol],
    "LargestWeight": [single_index_weights.max(), historical_mu_weights.max()],
    "TurnoverVsBase": [0.0, expected_return_turnover],
})

display(Table_P6_Risk_Aversion_Sensitivity)
display(Table_P7_Expected_Return_Sensitivity)
print(f"Risk-model allocation turnover: {risk_model_turnover:.2%}")
print(f"Expected-return-specification turnover: {expected_return_turnover:.2%}")

### Concentration-limit sensitivity

The stock and ETF caps are policy choices rather than case facts. Their purpose is to control concentration and reduce the influence of estimation error. The recommendation is therefore re-optimised under nearby cap combinations. A robust decision should not depend on one narrowly chosen pair of limits.


In [ ]:
# ============================================================
# 04E-2. CONCENTRATION-LIMIT SENSITIVITY
# ============================================================

cap_scenarios = [
    ("Tighter", 0.10, 0.30),
    ("Base", MAX_STOCK_WEIGHT, MAX_ETF_WEIGHT),
    ("Looser", 0.20, 0.40),
]

cap_rows = []
cap_weight_rows = []

for label, stock_cap, etf_cap in cap_scenarios:
    bounds = [
        (0.0, etf_cap if ticker in ETFS else stock_cap)
        for ticker in CANDIDATE_TICKERS
    ]
    if sum(b[1] for b in bounds) < 1.0 - 1e-12:
        raise ValueError(f"Infeasible concentration limits in scenario {label}.")

    w = optimize_mean_variance_utility(
        mu_base, SINGLE_INDEX_COV.values, BASE_RISK_AVERSION, bounds
    )
    er, vol, var = portfolio_statistics(w, mu_base, SINGLE_INDEX_COV.values)
    turnover_vs_base = 0.5 * np.abs(w - single_index_weights).sum()

    cap_rows.append({
        "Scenario": label,
        "StockCap": stock_cap,
        "ETFCap": etf_cap,
        "ExpectedReturn": er,
        "Volatility": vol,
        "Utility_A4": er - 0.5 * BASE_RISK_AVERSION * var,
        "LargestWeight": w.max(),
        "TurnoverVsBaseWeights": turnover_vs_base,
    })
    for ticker, weight in zip(CANDIDATE_TICKERS, w):
        cap_weight_rows.append({
            "Scenario": label,
            "Ticker": ticker,
            "Weight": weight,
        })

Table_P7B_Concentration_Sensitivity = pd.DataFrame(cap_rows)
Table_P7C_Concentration_Sensitivity_Weights = pd.DataFrame(cap_weight_rows)

display(Table_P7B_Concentration_Sensitivity)
display(Table_P7C_Concentration_Sensitivity_Weights)

Table_P7B_Concentration_Sensitivity.to_csv(
    TABLE_DIR / "Table_P7B_Concentration_Sensitivity.csv", index=False
)
Table_P7C_Concentration_Sensitivity_Weights.to_csv(
    TABLE_DIR / "Table_P7C_Concentration_Sensitivity_Weights.csv", index=False
)


In [ ]:
# DYNAMIC CONCENTRATION-SENSITIVITY INTERPRETATION
cap_table = Table_P7B_Concentration_Sensitivity.set_index("Scenario")
utility_range = cap_table["Utility_A4"].max() - cap_table["Utility_A4"].min()
max_turnover = cap_table["TurnoverVsBaseWeights"].max()

print(
    f"Concentration sensitivity: moving from the tighter to the looser cap specification "
    f"changes A=4 utility by {utility_range:.3%} across the tested scenarios; the largest "
    f"turnover from the base weights is {max_turnover:.1%}. These results indicate how much "
    "of the recommendation is attributable to the chosen concentration limits rather than the "
    "underlying return and risk estimates."
)


### Rejected alternative

A historical maximum-Sharpe allocation is used as the rejected alternative because it closely represents the treasurer’s proposal to favour the portfolio with the strongest historical risk-adjusted performance. It is evaluated using the same candidate universe and pre-2023 data so that the comparison reflects a difference in decision rule rather than a difference in information set.


In [ ]:
# ============================================================
# 04F. RECOMMENDED STRATEGY VS CREDIBLE REJECTED ALTERNATIVE
# ============================================================

# Eta's claim is operationalized as the strongest in-sample historical
# risk-adjusted portfolio: a historical-mean / sample-covariance max-Sharpe portfolio.
historical_mu = candidate_returns.mean().values * 12
historical_rf = rf_monthly.mean() * 12
eta_weights = optimize_max_sharpe(
    historical_mu, SAMPLE_COV.values, historical_rf, BASE_BOUNDS
)

strategy_weights = {
    "Utility_SampleCov": sample_weights,
    "Utility_SingleIndex": single_index_weights,
    "Eta_Historical_MaxSharpe": eta_weights,
}

# Initial policy choice. The single-index specification is used as the primary, parsimonious
# risk model; the sample-covariance portfolio is retained as the required robustness comparison.
# This is the strategy to freeze in the 22-Oct decision record.
INITIAL_GROWTH_STRATEGY = "Utility_SingleIndex"
CREDIBLE_REJECTED_ALTERNATIVE = "Eta_Historical_MaxSharpe"
initial_growth_weights = strategy_weights[INITIAL_GROWTH_STRATEGY]

# Common-evaluation lens: base expected returns + single-index covariance.
decision_rows = []
for strategy, weights in strategy_weights.items():
    er, vol, var = portfolio_statistics(weights, mu_base, SINGLE_INDEX_COV.values)
    decision_rows.append({
        "Strategy": strategy,
        "ExpectedReturn_CommonLens": er,
        "Volatility_CommonLens": vol,
        "Sharpe_CommonLens": (er - BASE_RF_ANNUAL) / vol,
        "Utility_A4_CommonLens": er - 0.5 * BASE_RISK_AVERSION * var,
        "LargestWeight": weights.max(),
        "DecisionStatus": (
            "Recommended initial growth policy" if strategy == INITIAL_GROWTH_STRATEGY
            else "Credible rejected alternative" if strategy == CREDIBLE_REJECTED_ALTERNATIVE
            else "Required model-comparison portfolio"
        ),
    })

Table_P8_Decision_Candidates = pd.DataFrame(decision_rows)
Table_P9_Strategy_Weights = pd.DataFrame({
    "Ticker": CANDIDATE_TICKERS,
    **{name: weights for name, weights in strategy_weights.items()},
})

display(Table_P8_Decision_Candidates)
display(Table_P9_Strategy_Weights)

### Decision robustness across risk models

No single portfolio can be expected to maximize utility under two different covariance specifications. The comparison below therefore evaluates the same three frozen candidate strategies under both risk models using the same expected-return vector and the same base risk aversion. This separates the choice of portfolio from the choice of risk model and makes the model dependence of the recommendation explicit.


In [ ]:
# ============================================================
# 04F-2. CROSS-RISK-MODEL DECISION ROBUSTNESS
# ============================================================

cross_model_rows = []
for lens_name, covariance in {
    "Sample covariance": SAMPLE_COV.values,
    "Single-index covariance": SINGLE_INDEX_COV.values,
}.items():
    lens_results = []
    for strategy, weights in strategy_weights.items():
        er, vol, var = portfolio_statistics(weights, mu_base, covariance)
        utility = er - 0.5 * BASE_RISK_AVERSION * var
        lens_results.append((strategy, utility))
        cross_model_rows.append({
            "RiskModelLens": lens_name,
            "Strategy": strategy,
            "ExpectedReturn": er,
            "Volatility": vol,
            "Utility_A4": utility,
            "Sharpe": (er - BASE_RF_ANNUAL) / vol,
        })

Table_P8B_Cross_Model_Decision_Robustness = pd.DataFrame(cross_model_rows)
preferred_by_lens = (
    Table_P8B_Cross_Model_Decision_Robustness
    .sort_values(["RiskModelLens", "Utility_A4"], ascending=[True, False])
    .groupby("RiskModelLens", as_index=False)
    .first()[["RiskModelLens", "Strategy", "Utility_A4"]]
    .rename(columns={"Strategy": "HighestUtilityStrategy"})
)

display(Table_P8B_Cross_Model_Decision_Robustness)
display(preferred_by_lens)

Table_P8B_Cross_Model_Decision_Robustness.to_csv(
    TABLE_DIR / "Table_P8B_Cross_Model_Decision_Robustness.csv", index=False
)

print(
    "Decision robustness: the preferred strategy is model-dependent. "
    "The initial policy uses the single-index covariance model as the primary risk specification "
    "because it imposes a parsimonious market-plus-idiosyncratic structure, while the sample-covariance "
    "solution is retained as a robustness benchmark rather than treated as evidence of a universally "
    "superior portfolio."
)


In [ ]:
# DYNAMIC PORTFOLIO DECISION INTERPRETATION
decision_table = Table_P8_Decision_Candidates.set_index("Strategy")
rec = decision_table.loc[INITIAL_GROWTH_STRATEGY]
eta = decision_table.loc[CREDIBLE_REJECTED_ALTERNATIVE]

utility_gap = rec["Utility_A4_CommonLens"] - eta["Utility_A4_CommonLens"]

print(
    f"Portfolio decision: {INITIAL_GROWTH_STRATEGY} is retained as the recommended growth allocation. "
    f"Under the common A={BASE_RISK_AVERSION} evaluation lens, its expected return is "
    f"{rec['ExpectedReturn_CommonLens']:.2%}, estimated volatility is {rec['Volatility_CommonLens']:.2%}, "
    f"and utility is {rec['Utility_A4_CommonLens']:.2%}. "
    f"The Eta alternative has utility {eta['Utility_A4_CommonLens']:.2%}, "
    f"giving the recommended strategy a utility advantage of {utility_gap:.2%}."
)


### Liability-funding logic

The two mandatory payments are separated from the growth objective. The two-year zero-coupon bond is used for the 2024 payment because its maturity date matches the obligation directly. The five-year Delta bond is used for the 2027 payment, and all interim coupons are retained for that obligation under a 0% reinvestment assumption. This is deliberately conservative because the funding plan does not depend on earning an additional return on the coupons.


In [ ]:
# ============================================================
# 04G. FULL USD 12M INITIAL ALLOCATION
# ============================================================

# The residual budget must cover both growth holdings and initial trading cost.
GROWTH_INVESTED_NOTIONAL = GROWTH_BUDGET_BEFORE_COSTS / (1 + BASE_TRADING_COST)
INITIAL_TRADING_COST = GROWTH_INVESTED_NOTIONAL * BASE_TRADING_COST
growth_dollar_allocations = initial_growth_weights * GROWTH_INVESTED_NOTIONAL

allocation_rows = [
    {
        "Sleeve": "2024 liability",
        "SecurityOrUse": "Beta",
        "DollarAllocation": beta_initial_cost,
        "Role": "Fund 31-Dec-2024 scholarship payment",
    },
    {
        "Sleeve": "2027 liability",
        "SecurityOrUse": "Delta",
        "DollarAllocation": delta_initial_cost,
        "Role": "Fund 31-Dec-2027 learning-centre payment",
    },
]

for ticker, amount in zip(CANDIDATE_TICKERS, growth_dollar_allocations):
    allocation_rows.append({
        "Sleeve": "Growth",
        "SecurityOrUse": ticker,
        "DollarAllocation": amount,
        "Role": "Pursue 2030 real-value objective",
    })

allocation_rows.append({
    "Sleeve": "Implementation",
    "SecurityOrUse": "Initial trading cost",
    "DollarAllocation": INITIAL_TRADING_COST,
    "Role": f"{BASE_TRADING_COST:.2%} one-way cost on initial growth purchases",
})

Table_P10_Full_Allocation = pd.DataFrame(allocation_rows)
Table_P10_Full_Allocation["PercentOfInitialFund"] = (
    Table_P10_Full_Allocation["DollarAllocation"] / INITIAL_CAPITAL
)

allocation_total = Table_P10_Full_Allocation["DollarAllocation"].sum()
if not np.isclose(allocation_total, INITIAL_CAPITAL, atol=0.01):
    raise ValueError("Initial allocation does not reconcile to USD 12 million.")

display(Table_P10_Full_Allocation)
print(f"Full initial use of capital: USD {allocation_total:,.2f}")

### Frozen out-of-sample evaluation

The growth strategy is evaluated from 2023 to 2025 using the policy fixed at the 2022 decision date. No future data are used to re-estimate expected returns, covariances or target weights. This preserves the separation between the initial decision and subsequent evidence and allows a fair comparison with the frozen SPY benchmark.


In [ ]:
# ============================================================
# 04H. TRUSTEE 15% LOSS CONCERN AND FROZEN IMPLEMENTATION RULE
# ============================================================

recommended_row = Table_P8_Decision_Candidates.loc[
    Table_P8_Decision_Candidates["Strategy"] == INITIAL_GROWTH_STRATEGY
].iloc[0]

# Diagnostic only: use a one-year normal approximation for the growth sleeve.
# The case does not define a horizon or probability for the 15% concern.
z_5pct = norm.ppf(0.05)
growth_5pct_return = (
    recommended_row["ExpectedReturn_CommonLens"] + z_5pct * recommended_row["Volatility_CommonLens"]
)
growth_share_of_total = GROWTH_INVESTED_NOTIONAL / INITIAL_CAPITAL
approx_total_fund_5pct_growth_impact = growth_share_of_total * growth_5pct_return

# Historical diagnostics under the frozen target weights provide a second lens that does
# not assume normality. This is still retrospective evidence, not a guaranteed loss bound.
estimation_growth_proxy = pd.Series(
    returns_for_model[CANDIDATE_TICKERS].values @ initial_growth_weights,
    index=returns_for_model.index,
)
rolling_12m = (1 + estimation_growth_proxy).rolling(12).apply(np.prod, raw=True) - 1
estimation_wealth = (1 + estimation_growth_proxy).cumprod()
estimation_drawdown = estimation_wealth / estimation_wealth.cummax() - 1
worst_12m_return = rolling_12m.min()
max_drawdown = estimation_drawdown.min()

Table_P11_Risk_Tolerance = pd.DataFrame({
    "Metric": [
        "Trustee reference loss level",
        "Growth share of initial fund",
        "Approximate 5% one-year growth-sleeve return",
        "Approximate growth-only contribution to total-fund return",
        "Worst rolling 12-month growth return, 2015-2022",
        "Maximum growth drawdown, 2015-2022",
        "Difference versus -15% reference",
    ],
    "Value": [
        -0.15,
        growth_share_of_total,
        growth_5pct_return,
        approx_total_fund_5pct_growth_impact,
        worst_12m_return,
        max_drawdown,
        approx_total_fund_5pct_growth_impact - (-0.15),
    ],
    "Interpretation": [
        "Reference concern only; not a formal hard limit",
        "Fixed-payment reserve is separated before growth allocation",
        "Normal approximation; diagnostic, not guarantee",
        "Growth sleeve only; this is not a complete total-fund downside measure because bond mark-to-market risk is assessed separately in Section 02",
        "Historical diagnostic; retrospective and not a future guarantee",
        "Historical diagnostic; retrospective and not a future guarantee",
        "Diagnostic comparison only; the trustee statement does not define a formal horizon or probability",
    ],
})

Table_P12_Implementation_Policy = pd.DataFrame({
    "PolicyItem": [
        "Liability priority",
        "Frozen initial growth strategy",
        "Benchmark",
        "Rebalancing schedule",
        "Rebalancing rule",
        "Re-estimation during 2023-2025 evaluation",
        "Rebalancing trading cost",
        "Maximum stock weight",
        "Maximum ETF weight",
        "Borrowing / shorting / derivatives",
        "Status after 22-Oct submission",
    ],
    "Policy": [
        "Fund mandatory payments before allocating growth capital",
        INITIAL_GROWTH_STRATEGY,
        BENCHMARK,
        "Annually on 31 December; make any scheduled liability payment first",
        "After the liability payment, rebalance the remaining growth sleeve back to the original frozen target weights",
        "No re-estimation or re-optimization; future data cannot rewrite the initial decision",
        f"{BASE_TRADING_COST:.2%} one-way on traded notional",
        f"{MAX_STOCK_WEIGHT:.0%}",
        f"{MAX_ETF_WEIGHT:.0%}",
        "Prohibited",
        "Frozen policy; later evidence is compared against it and recorded in the decision log",
    ],
})

display(Table_P11_Risk_Tolerance)
display(Table_P12_Implementation_Policy)


In [ ]:
# ============================================================
# 04I. MATERIAL OUTPUTS ONLY — EXPORT FOR MEMO / SLIDES
# ============================================================

material_outputs = {
    "Table_A1_Mandate_Assumptions": Table_A1_Mandate_Assumptions,
    "Table_B2_Bond_Analytics": Table_B2_Bond_Analytics,
    "Table_B3_Duration_Convexity_Check": Table_B3_Duration_Convexity_Check,
    "Table_B4_Funding_Plan": Table_B4_Funding_Plan,
    "Table_B6_Rate_Scenarios": Table_B6_Rate_Scenarios,
    "Table_B7_Liability_Sleeve_Under_Rate_Stress": Table_B7_Liability_Sleeve_Under_Rate_Stress,
    "Table_D1_Candidate_Universe": Table_D1_Candidate_Universe,
    "Table_D2_Eligibility_Evidence": Table_D2_Eligibility_Evidence,
    "Table_D4_Return_Construction": Table_D4_Return_Construction,
    "Table_P1_French_Source_Record": Table_P1_French_Source_Record,
    "Table_P2_Expected_Return_Inputs": Table_P2_Expected_Return_Inputs.reset_index(),
    "Table_P3_Constraints": Table_P3_Constraints,
    "Table_P4_Risk_Model_Comparison": Table_P4_Risk_Model_Comparison,
    "Table_P5_Risk_Model_Summary": Table_P5_Risk_Model_Summary,
    "Table_P6_Risk_Aversion_Sensitivity": Table_P6_Risk_Aversion_Sensitivity,
    "Table_P7_Expected_Return_Sensitivity": Table_P7_Expected_Return_Sensitivity,
    "Table_P8_Decision_Candidates": Table_P8_Decision_Candidates,
    "Table_P9_Strategy_Weights": Table_P9_Strategy_Weights,
    "Table_P10_Full_Allocation": Table_P10_Full_Allocation,
    "Table_P11_Risk_Tolerance": Table_P11_Risk_Tolerance,
    "Table_P12_Implementation_Policy": Table_P12_Implementation_Policy,
}

for name, table in material_outputs.items():
    table.to_csv(TABLE_DIR / f"{name}.csv", index=False)

# Matrices are saved for reproducibility but are not displayed as decision evidence.
SAMPLE_COV.to_csv(TABLE_DIR / "Matrix_Sample_Covariance.csv")
SINGLE_INDEX_COV.to_csv(TABLE_DIR / "Matrix_Single_Index_Covariance.csv")

print("Sections 01–04 requirement-matched outputs exported.")
print(f"Recommended initial growth strategy: {INITIAL_GROWTH_STRATEGY}")
print(f"Credible rejected alternative: {CREDIBLE_REJECTED_ALTERNATIVE}")


# 5. Market and Factor Explanations

The growth allocation is evaluated using two return models from Lecture 7:

1. a market-only CAPM; and
2. the Fama–French three-factor model.

The comparison is performed for the recommended growth portfolio and for two representative securities, AAPL and JPM.

The purpose is to identify the economic sources of return and risk rather than to treat estimated alpha as a forecast. Market beta measures broad equity-market exposure, while SMB and HML capture size and value tilts. Any estimated alpha is interpreted together with its statistical uncertainty and is not assumed to persist out of sample.


### Factor-data construction

Fama–French monthly factors are used in two places: the market excess return supports the CAPM-style expected-return inputs, and the FF3 model is used later to explain the economic exposures of the growth portfolio. Factor observations are aligned by month with the market-return data so that all regressions use a common sample.


In [ ]:
# ============================================================
# 05A. MARKET-ONLY VS FAMA-FRENCH 3-FACTOR EXPLANATIONS
# ============================================================

# Required FF3 columns from the saved Kenneth French dataset.
REQUIRED_FF3_COLUMNS = ["Mkt-RF", "SMB", "HML", "RF"]
missing_ff3 = [c for c in REQUIRED_FF3_COLUMNS if c not in factors_for_model.columns]
if missing_ff3:
    raise ValueError(
        f"French factor file is missing required FF3 columns: {missing_ff3}. "
        "Use the Fama-French Research Data Factors monthly dataset."
    )

# At least two individual securities are required by the assignment.
FACTOR_TEST_SECURITIES = ["AAPL", "JPM"]
missing_test_assets = [t for t in FACTOR_TEST_SECURITIES if t not in returns_for_model.columns]
if missing_test_assets:
    raise ValueError(f"Missing factor-test securities: {missing_test_assets}")

# ------------------------------------------------------------
# 1. Reconstruct the recommended growth portfolio historically
# ------------------------------------------------------------
# Reconstruct the recommended portfolio using the same annual end-December
# rebalancing rule as the frozen policy. Gross returns are used here because
# the regression is intended to explain market exposures rather than implementation costs.
def frozen_annual_rebalance_returns(monthly_asset_returns, target_weights, rebalance_month=12):
    R = monthly_asset_returns.copy().astype(float)
    target = np.asarray(target_weights, dtype=float)
    weights = target.copy()
    out = []
    for dt, r_series in R.iterrows():
        r = r_series.values.astype(float)
        gross_return = float(weights @ r)
        end_values = weights * (1 + r)
        end_weights = end_values / end_values.sum()
        out.append(gross_return)
        weights = target.copy() if dt.month == rebalance_month else end_weights
    return pd.Series(out, index=R.index)

growth_portfolio_return = frozen_annual_rebalance_returns(
    returns_for_model[CANDIDATE_TICKERS],
    initial_growth_weights,
).rename("Recommended_Growth_Portfolio")

factor_test_returns = pd.DataFrame(
    {
        "Recommended Growth Portfolio": growth_portfolio_return,
        **{
            ticker: returns_for_model[ticker]
            for ticker in FACTOR_TEST_SECURITIES
        },
    }
)

# Ensure one common sample for every model and every tested asset.
factor_analysis_data = pd.concat(
    [
        factor_test_returns,
        factors_for_model[REQUIRED_FF3_COLUMNS],
    ],
    axis=1,
    join="inner",
).dropna()

if len(factor_analysis_data) < 60:
    raise ValueError("Fewer than 60 common monthly observations for factor analysis.")
if factor_analysis_data.index.max() > DECISION_DATE:
    raise ValueError("Post-2022 information leaked into factor analysis.")

print(
    f"Factor-analysis sample: {factor_analysis_data.index.min():%Y-%m} "
    f"to {factor_analysis_data.index.max():%Y-%m} "
    f"({len(factor_analysis_data)} monthly observations)"
)

# ------------------------------------------------------------
# 2. Regression helper
# ------------------------------------------------------------
def fit_factor_model(asset_name, model_name, factor_columns):
    """
    Regress monthly asset excess return on the selected factor set.
    Returns the fitted model and one compact result row.
    """
    y = (
        factor_analysis_data[asset_name]
        - factor_analysis_data["RF"]
    )

    X = sm.add_constant(
        factor_analysis_data[factor_columns],
        has_constant="add",
    )

    model = sm.OLS(y, X).fit()

    alpha_monthly = model.params["const"]
    alpha_annualized = alpha_monthly * 12

    alpha_ci = model.conf_int(alpha=0.05).loc["const"] * 12

    row = {
        "Asset": asset_name,
        "Model": model_name,
        "N": int(model.nobs),
        "AlphaMonthly": alpha_monthly,
        "AlphaAnnualized": alpha_annualized,
        "AlphaPValue": model.pvalues["const"],
        "Alpha95CI_Lower_Annual": alpha_ci.iloc[0],
        "Alpha95CI_Upper_Annual": alpha_ci.iloc[1],
        "MarketBeta": model.params.get("Mkt-RF", np.nan),
        "MarketBetaPValue": model.pvalues.get("Mkt-RF", np.nan),
        "SMBLoading": model.params.get("SMB", np.nan),
        "SMBPValue": model.pvalues.get("SMB", np.nan),
        "HMLLoading": model.params.get("HML", np.nan),
        "HMLPValue": model.pvalues.get("HML", np.nan),
        "R2": model.rsquared,
        "AdjustedR2": model.rsquared_adj,
    }

    return model, row


# ------------------------------------------------------------
# 3. Estimate CAPM and FF3 for portfolio + two securities
# ------------------------------------------------------------
factor_models = {}
factor_rows = []

for asset_name in factor_test_returns.columns:
    capm_model, capm_row = fit_factor_model(
        asset_name=asset_name,
        model_name="CAPM",
        factor_columns=["Mkt-RF"],
    )
    factor_models[(asset_name, "CAPM")] = capm_model
    factor_rows.append(capm_row)

    ff3_model, ff3_row = fit_factor_model(
        asset_name=asset_name,
        model_name="FF3",
        factor_columns=["Mkt-RF", "SMB", "HML"],
    )
    factor_models[(asset_name, "FF3")] = ff3_model
    factor_rows.append(ff3_row)

Table_F1_Factor_Model_Results = pd.DataFrame(factor_rows)

# Clean ordering for memo/slide use.
Table_F1_Factor_Model_Results = Table_F1_Factor_Model_Results[
    [
        "Asset",
        "Model",
        "N",
        "AlphaAnnualized",
        "AlphaPValue",
        "Alpha95CI_Lower_Annual",
        "Alpha95CI_Upper_Annual",
        "MarketBeta",
        "MarketBetaPValue",
        "SMBLoading",
        "SMBPValue",
        "HMLLoading",
        "HMLPValue",
        "AdjustedR2",
        "R2",
        "AlphaMonthly",
    ]
]

display(Table_F1_Factor_Model_Results)


# ============================================================
# 05B. MODEL COMPARISON AND DECISION-RELEVANT INTERPRETATION
# ============================================================

comparison_rows = []

for asset_name in factor_test_returns.columns:
    capm = Table_F1_Factor_Model_Results[
        (Table_F1_Factor_Model_Results["Asset"] == asset_name)
        & (Table_F1_Factor_Model_Results["Model"] == "CAPM")
    ].iloc[0]

    ff3 = Table_F1_Factor_Model_Results[
        (Table_F1_Factor_Model_Results["Asset"] == asset_name)
        & (Table_F1_Factor_Model_Results["Model"] == "FF3")
    ].iloc[0]

    comparison_rows.append(
        {
            "Asset": asset_name,
            "CAPM_AdjustedR2": capm["AdjustedR2"],
            "FF3_AdjustedR2": ff3["AdjustedR2"],
            "ChangeInAdjustedR2": ff3["AdjustedR2"] - capm["AdjustedR2"],
            "CAPM_AlphaAnnualized": capm["AlphaAnnualized"],
            "CAPM_AlphaPValue": capm["AlphaPValue"],
            "FF3_AlphaAnnualized": ff3["AlphaAnnualized"],
            "FF3_AlphaPValue": ff3["AlphaPValue"],
            "FF3_MarketBeta": ff3["MarketBeta"],
            "FF3_SMBLoading": ff3["SMBLoading"],
            "FF3_HMLLoading": ff3["HMLLoading"],
            "FF3_AlphaStatisticallySignificant_5pct": bool(
                ff3["AlphaPValue"] < 0.05
            ),
        }
    )

Table_F2_CAPM_vs_FF3 = pd.DataFrame(comparison_rows)
display(Table_F2_CAPM_vs_FF3)


# ------------------------------------------------------------
# 4. Compact exposure labels for interpretation
# ------------------------------------------------------------
def loading_label(value, threshold=0.10):
    if pd.isna(value):
        return "Not estimated"
    if value > threshold:
        return "Positive"
    if value < -threshold:
        return "Negative"
    return "Near zero"


interpretation_rows = []

for _, row in Table_F2_CAPM_vs_FF3.iterrows():
    interpretation_rows.append(
        {
            "Asset": row["Asset"],
            "MarketExposure": loading_label(row["FF3_MarketBeta"]),
            "SMBExposure": loading_label(row["FF3_SMBLoading"]),
            "HMLExposure": loading_label(row["FF3_HMLLoading"]),
            "DoesFF3ImproveAdjustedR2": bool(row["ChangeInAdjustedR2"] > 0),
            "FF3AlphaSignificantAt5pct": bool(
                row["FF3_AlphaPValue"] < 0.05
            ),
            "DecisionInterpretation": (
                "Factor exposures help explain historical excess returns; "
                "alpha is not treated as persistent skill."
                if row["FF3_AlphaPValue"] >= 0.05
                else
                "Historical alpha is statistically distinguishable from zero in this sample, "
                "but it is still not assumed to persist out of sample."
            ),
        }
    )

Table_F3_Factor_Interpretation = pd.DataFrame(interpretation_rows)
display(Table_F3_Factor_Interpretation)


# ------------------------------------------------------------
# 5. Save traceable outputs
# ------------------------------------------------------------
for name, table in {
    "Table_F1_Factor_Model_Results": Table_F1_Factor_Model_Results,
    "Table_F2_CAPM_vs_FF3": Table_F2_CAPM_vs_FF3,
    "Table_F3_Factor_Interpretation": Table_F3_Factor_Interpretation,
}.items():
    table.to_csv(TABLE_DIR / f"{name}.csv", index=False)

print(
    "\nSection 05 interpretation rule:\n"
    "- CAPM asks how much broad-market exposure explains.\n"
    "- FF3 tests whether size (SMB) and value (HML) add explanatory power.\n"
    "- Compare adjusted R² rather than treating extra factors as automatically better.\n"
    "- Alpha must be read together with its p-value and confidence interval.\n"
    "- Even a statistically significant historical alpha does not establish persistent future abnormal return."
)


**Interpretation.** The factor regressions are used to explain the portfolio’s return drivers, not to justify the portfolio through historical alpha. Additional explanatory power from SMB and HML indicates that common factor exposures account for part of the return pattern beyond the market factor alone. Any statistically significant alpha remains sample-specific and does not establish persistent abnormal performance.


In [ ]:
# DYNAMIC FACTOR-MODEL INTERPRETATION
factor_cmp = Table_F2_CAPM_vs_FF3.set_index("Asset")
factor_res = Table_F1_Factor_Model_Results.set_index(["Asset", "Model"])

rec_cmp = factor_cmp.loc["Recommended Growth Portfolio"]
rec_ff3 = factor_res.loc[("Recommended Growth Portfolio", "FF3")]
aapl_ff3 = factor_res.loc[("AAPL", "FF3")]
jpm_ff3 = factor_res.loc[("JPM", "FF3")]

print(
    "Factor-model interpretation: "
    f"for the recommended growth portfolio, FF3 changes adjusted R² from "
    f"{rec_cmp['CAPM_AdjustedR2']:.3f} under CAPM to {rec_cmp['FF3_AdjustedR2']:.3f}. "
    f"The FF3 alpha is {rec_ff3['AlphaAnnualized']:.2%} annualized "
    f"(p={rec_ff3['AlphaPValue']:.3f}). "
    f"AAPL and JPM have FF3 alpha p-values of {aapl_ff3['AlphaPValue']:.3f} "
    f"and {jpm_ff3['AlphaPValue']:.3f}, respectively. "
    "These regressions are used to explain exposures; estimated alpha is not included "
    "in the forward expected-return inputs and is not treated as evidence of persistent skill."
)


# 6. Out-of-Sample Evaluation, 2023–2025

The initial growth strategy is frozen at the 31 December 2022 decision date and evaluated over January 2023 to December 2025 without re-optimisation using future information.

The recommended strategy is compared with SPY using cumulative return, annualised return, volatility, Sharpe ratio and maximum drawdown. This evaluation is used as evidence against or in support of the original decision; it is not used to rewrite the initial policy retrospectively.

The out-of-sample results are intentionally reported even where they are unfavourable to the recommendation.


### Frozen out-of-sample evaluation

The growth strategy is evaluated from 2023 to 2025 using the policy fixed at the 2022 decision date. No future data are used to re-estimate expected returns, covariances or target weights. This preserves the separation between the initial decision and subsequent evidence and allows a fair comparison with the frozen SPY benchmark.


In [ ]:
# ============================================================
# 06A. FROZEN STRATEGY VS FROZEN BENCHMARK, 2023-2025
# ============================================================

def simulate_frozen_strategy(monthly_asset_returns, target_weights, trading_cost=0.0,
                             rebalance_month=12):
    """Annual end-December rebalance to frozen target weights; no re-estimation."""
    R = monthly_asset_returns.copy().astype(float)
    target = np.asarray(target_weights, dtype=float)
    if not np.isclose(target.sum(), 1.0, atol=1e-10):
        raise ValueError("Target weights must sum to 1.")
    if (target < -1e-12).any():
        raise ValueError("Negative target weight found in long-only strategy.")

    weights = target.copy()
    rows = []
    for dt, r_series in R.iterrows():
        r = r_series.values.astype(float)
        gross_return = float(weights @ r)
        end_values = weights * (1 + r)
        end_weights = end_values / end_values.sum()

        gross_traded_fraction = 0.0
        cost_fraction = 0.0
        rebalance = (dt.month == rebalance_month and dt != R.index[-1])
        if rebalance:
            gross_traded_fraction = float(np.abs(target - end_weights).sum())
            cost_fraction = trading_cost * gross_traded_fraction
            next_weights = target.copy()
        else:
            next_weights = end_weights

        net_return = (1 + gross_return) * (1 - cost_fraction) - 1
        rows.append({
            "Date": dt,
            "GrossReturn": gross_return,
            "NetReturn": net_return,
            "GrossTradedFraction": gross_traded_fraction,
            "TradingCostFraction": cost_fraction,
        })
        weights = next_weights

    result = pd.DataFrame(rows).set_index("Date")
    return result


eval_assets = evaluation_returns.loc[:, CANDIDATE_TICKERS].copy()
if len(eval_assets) != 36:
    print(f"Warning: expected 36 evaluation months; found {len(eval_assets)} common complete months.")

recommended_oos = simulate_frozen_strategy(
    eval_assets,
    initial_growth_weights,
    trading_cost=BASE_TRADING_COST,
    rebalance_month=12,
)
benchmark_oos = pd.DataFrame(index=eval_assets.index)
benchmark_oos["NetReturn"] = eval_assets[BENCHMARK]

# Evaluation-period RF must come from a saved French file; do not reuse the 2015-2022 RF.
if not FRENCH_EVALUATION_FILE.exists():
    raise FileNotFoundError(
        "Missing processed French evaluation factors for 2023-2025. "
        "Run the retrieval cell once with RUN_DATA_RETRIEVAL=True, then rerun offline."
    )
french_evaluation = pd.read_csv(FRENCH_EVALUATION_FILE, index_col="Date", parse_dates=True)
common_eval_dates = recommended_oos.index.intersection(french_evaluation.index)
if len(common_eval_dates) < 30:
    raise ValueError("Insufficient common 2023-2025 months for out-of-sample evaluation.")


def performance_summary(name, returns, rf):
    r = returns.loc[common_eval_dates].astype(float)
    rf_aligned = rf.loc[common_eval_dates].astype(float)
    wealth = (1 + r).cumprod()
    n_months = len(r)
    ann_return = wealth.iloc[-1] ** (12 / n_months) - 1
    ann_vol = r.std(ddof=1) * np.sqrt(12)
    ann_excess_arith = (r - rf_aligned).mean() * 12
    sharpe = ann_excess_arith / ann_vol if ann_vol > 0 else np.nan
    drawdown = wealth / wealth.cummax() - 1
    return {
        "Strategy": name,
        "Months": n_months,
        "CumulativeReturn": wealth.iloc[-1] - 1,
        "AnnualizedReturn": ann_return,
        "AnnualizedVolatility": ann_vol,
        "Sharpe_UsingEvaluationRF": sharpe,
        "MaxDrawdown": drawdown.min(),
        "EndingWealthPerDollar": wealth.iloc[-1],
    }

Table_E1_Frozen_Evaluation = pd.DataFrame([
    performance_summary(
        INITIAL_GROWTH_STRATEGY,
        recommended_oos["NetReturn"],
        french_evaluation["RF"],
    ),
    performance_summary(
        f"Benchmark_{BENCHMARK}",
        benchmark_oos["NetReturn"],
        french_evaluation["RF"],
    ),
])

display(Table_E1_Frozen_Evaluation)

# Rebalancing-cost evidence.
Table_E2_Rebalancing_Costs = recommended_oos.loc[
    recommended_oos["GrossTradedFraction"] > 0,
    ["GrossTradedFraction", "TradingCostFraction"]
].reset_index()
display(Table_E2_Rebalancing_Costs)

# Cumulative wealth figure: one material chart only.
wealth_plot = pd.DataFrame({
    INITIAL_GROWTH_STRATEGY: (1 + recommended_oos["NetReturn"]).cumprod(),
    BENCHMARK: (1 + benchmark_oos["NetReturn"]).cumprod(),
})
ax = wealth_plot.plot(figsize=(9, 5))
ax.set_title("Frozen Growth Strategy vs SPY: 2023–2025")
ax.set_xlabel("Date")
ax.set_ylabel("Growth of USD 1")
ax.grid(False)
fig = ax.get_figure()
fig.tight_layout()
fig.savefig(FIGURE_DIR / "Figure_E1_Frozen_Strategy_vs_SPY.png", dpi=160, bbox_inches="tight")
plt.show()

Table_E1_Frozen_Evaluation.to_csv(TABLE_DIR / "Table_E1_Frozen_Evaluation.csv", index=False)
Table_E2_Rebalancing_Costs.to_csv(TABLE_DIR / "Table_E2_Rebalancing_Costs.csv", index=False)


In [ ]:
# DYNAMIC OUT-OF-SAMPLE INTERPRETATION
eval_table = Table_E1_Frozen_Evaluation.set_index("Strategy")
rec_oos = eval_table.loc[INITIAL_GROWTH_STRATEGY]
bench_name = f"Benchmark_{BENCHMARK}"
spy_oos = eval_table.loc[bench_name]

outperformed = rec_oos["AnnualizedReturn"] > spy_oos["AnnualizedReturn"]
direction = "outperformed" if outperformed else "underperformed"

print(
    f"Out-of-sample result: the frozen recommended strategy {direction} {BENCHMARK} over 2023–2025. "
    f"Annualized return was {rec_oos['AnnualizedReturn']:.2%} versus {spy_oos['AnnualizedReturn']:.2%}; "
    f"cumulative return was {rec_oos['CumulativeReturn']:.2%} versus {spy_oos['CumulativeReturn']:.2%}; "
    f"Sharpe ratio was {rec_oos['Sharpe_UsingEvaluationRF']:.2f} versus "
    f"{spy_oos['Sharpe_UsingEvaluationRF']:.2f}; and maximum drawdown was "
    f"{rec_oos['MaxDrawdown']:.2%} versus {spy_oos['MaxDrawdown']:.2%}. "
    "The result is retained as evidence about the frozen growth allocation and is not used "
    "to rewrite the original 2022 decision."
)


In [ ]:
# OUT-OF-SAMPLE HORIZON LIMITATION
evaluation_years = len(common_eval_dates) / 12
print(
    f"Evaluation horizon: {len(common_eval_dates)} monthly observations "
    f"({evaluation_years:.1f} years). This period is informative about the frozen 2022 choice, "
    "but it is too short to establish long-run dominance and it does not test whether the "
    "eight-year 2030 real-value objective is ultimately achieved."
)


# 7. Real-Value Objective and Decision Sensitivity

The trustees’ long-term objective is to preserve the December 2022 purchasing power of USD 8 million by the end of 2030 after the mandatory payments have been met.

A 2.5% annual inflation assumption is used in the base case. This converts the real-value target into a nominal 2030 objective and allows the required compound growth rate of the residual investment to be calculated.

The analysis then tests how sensitive the recommendation is to a material change in the common market-risk-premium assumption. The threshold at which the recommended strategy becomes utility-indifferent to the rejected alternative is reported as the point at which the original decision would change.


### Why two covariance models are compared

The assignment requires the portfolio decision to be tested under both the sample covariance matrix and a single-index risk model. The sample covariance matrix preserves the full historical cross-sectional dependence, while the single-index model imposes a simpler structure in which co-movement is largely explained by market beta. Running the same optimisation under both risk estimates shows how dependent the recommended weights are on the covariance specification.


In [ ]:
# ============================================================
# 07A. REQUIRED CAGR FOR THE 2030 REAL-VALUE OBJECTIVE
# ============================================================
required_growth_cagr = (
    NOMINAL_TARGET_2030 / GROWTH_INVESTED_NOTIONAL
) ** (1 / 8) - 1

recommended_mu, recommended_vol, recommended_var = portfolio_statistics(
    initial_growth_weights, mu_base, SINGLE_INDEX_COV.values
)

Table_G1_2030_Goal_Feasibility = pd.DataFrame({
    "Metric": [
        "Initial growth capital after initial trading cost",
        "2030 nominal target under base inflation",
        "Required annual compound return",
        "Recommended portfolio arithmetic expected return",
        "Recommended portfolio annual volatility estimate",
        "Arithmetic expected-return difference versus required CAGR",
    ],
    "Value": [
        GROWTH_INVESTED_NOTIONAL,
        NOMINAL_TARGET_2030,
        required_growth_cagr,
        recommended_mu,
        recommended_vol,
        recommended_mu - required_growth_cagr,
    ],
    "Interpretation": [
        "Capital pursuing the 2030 mission after liabilities are reserved",
        "Planning target, not a promised terminal value",
        "Break-even compound rate needed to hit the nominalized real-value objective",
        "CAPM-style arithmetic forecast input; not a guaranteed CAGR",
        "Single-index annualized risk estimate",
        "Descriptive difference only; arithmetic expected return is not a compound success probability",
    ],
})
display(Table_G1_2030_Goal_Feasibility)

# Inflation sensitivity for the real target.
inflation_grid = np.array([0.020, 0.025, 0.030, 0.040])
Table_G2_Inflation_Sensitivity = pd.DataFrame({
    "InflationAssumption": inflation_grid,
    "Nominal2030Target": REAL_TARGET_2030 * (1 + inflation_grid) ** 8,
})
Table_G2_Inflation_Sensitivity["RequiredGrowthCAGR"] = (
    Table_G2_Inflation_Sensitivity["Nominal2030Target"] / GROWTH_INVESTED_NOTIONAL
) ** (1 / 8) - 1
Table_G2_Inflation_Sensitivity["ExpectedReturnMargin"] = (
    recommended_mu - Table_G2_Inflation_Sensitivity["RequiredGrowthCAGR"]
)
display(Table_G2_Inflation_Sensitivity)

# ============================================================
# 07B. EXPLICIT DECISION-REVERSAL THRESHOLD
# ============================================================
eta_fixed_weights = strategy_weights[CREDIBLE_REJECTED_ALTERNATIVE]
eta_mu, eta_vol, eta_var = portfolio_statistics(
    eta_fixed_weights, mu_base, SINGLE_INDEX_COV.values
)

# Because every CAPM-style asset forecast is RF + beta * market premium, changing
# the common market-premium assumption changes both frozen strategies consistently.
asset_betas = Table_P2_Expected_Return_Inputs.loc[CANDIDATE_TICKERS, "CAPMBeta"].values
recommended_beta = float(initial_growth_weights @ asset_betas)
eta_beta = float(eta_fixed_weights @ asset_betas)

if np.isclose(recommended_beta, eta_beta):
    raise ValueError("The two strategies have indistinguishable market beta; MRP threshold is undefined.")

# Solve U_rec = U_eta at A=4. The common RF cancels because both portfolios are fully invested.
mrp_indifference = (
    0.5 * BASE_RISK_AVERSION * (recommended_var - eta_var)
    / (recommended_beta - eta_beta)
)
mrp_change_to_flip = BASE_MARKET_PREMIUM_ANNUAL - mrp_indifference

Table_G3_Decision_Reversal_Threshold = pd.DataFrame({
    "Item": [
        "Base annual market risk premium",
        "Market risk premium at utility indifference",
        "Reduction in market risk premium that flips preference",
        "Recommended portfolio CAPM beta",
        "Eta portfolio CAPM beta",
        "Recommended variance under common lens",
        "Eta variance under common lens",
    ],
    "Value": [
        BASE_MARKET_PREMIUM_ANNUAL,
        mrp_indifference,
        mrp_change_to_flip,
        recommended_beta,
        eta_beta,
        recommended_var,
        eta_var,
    ],
})

if 0 < mrp_indifference < BASE_MARKET_PREMIUM_ANNUAL:
    reversal_text = (
        f"At A={BASE_RISK_AVERSION}, reducing the common annual market risk premium from "
        f"{BASE_MARKET_PREMIUM_ANNUAL:.2%} to approximately {mrp_indifference:.2%} makes the "
        "frozen recommended strategy and Eta alternative utility-indifferent; a lower premium "
        "reverses the ranking in favour of Eta under this common lens."
    )
else:
    reversal_text = (
        "The market-risk-premium threshold does not lie between zero and the base assumption; "
        "review the reversal specification before submission."
    )

print(reversal_text)
display(Table_G3_Decision_Reversal_Threshold)

for name, table in {
    "Table_G1_2030_Goal_Feasibility": Table_G1_2030_Goal_Feasibility,
    "Table_G2_Inflation_Sensitivity": Table_G2_Inflation_Sensitivity,
    "Table_G3_Decision_Reversal_Threshold": Table_G3_Decision_Reversal_Threshold,
}.items():
    table.to_csv(TABLE_DIR / f"{name}.csv", index=False)


### Robustness of the reversal threshold

The market-risk-premium threshold is a conditional result, not a forecast. It depends on the risk model and on the trustees’ assumed risk aversion. The threshold is therefore recalculated under both covariance specifications and across the same risk-aversion values used elsewhere in the notebook.


In [ ]:
# ============================================================
# 07C. REVERSAL-THRESHOLD ROBUSTNESS
# ============================================================

threshold_rows = []
for risk_model_name, covariance in {
    "Sample covariance": SAMPLE_COV.values,
    "Single-index covariance": SINGLE_INDEX_COV.values,
}.items():
    rec_er, rec_vol, rec_var = portfolio_statistics(
        initial_growth_weights, mu_base, covariance
    )
    eta_er, eta_vol, eta_var_local = portfolio_statistics(
        eta_fixed_weights, mu_base, covariance
    )

    for A in RISK_AVERSION_VALUES:
        denom = recommended_beta - eta_beta
        if np.isclose(denom, 0.0):
            threshold = np.nan
        else:
            threshold = 0.5 * A * (rec_var - eta_var_local) / denom

        rec_utility_at_base = (
            BASE_RF_ANNUAL + recommended_beta * BASE_MARKET_PREMIUM_ANNUAL
            - 0.5 * A * rec_var
        )
        eta_utility_at_base = (
            BASE_RF_ANNUAL + eta_beta * BASE_MARKET_PREMIUM_ANNUAL
            - 0.5 * A * eta_var_local
        )

        threshold_rows.append({
            "RiskModel": risk_model_name,
            "RiskAversionA": A,
            "BaseMarketRiskPremium": BASE_MARKET_PREMIUM_ANNUAL,
            "MRP_UtilityIndifference": threshold,
            "ChangeFromBaseMRP": BASE_MARKET_PREMIUM_ANNUAL - threshold,
            "PreferredAtBaseMRP": (
                INITIAL_GROWTH_STRATEGY
                if rec_utility_at_base >= eta_utility_at_base
                else CREDIBLE_REJECTED_ALTERNATIVE
            ),
            "ThresholdWithinZeroToBase": bool(
                np.isfinite(threshold)
                and threshold > 0
                and threshold < BASE_MARKET_PREMIUM_ANNUAL
            ),
        })

Table_G4_Reversal_Threshold_Robustness = pd.DataFrame(threshold_rows)
display(Table_G4_Reversal_Threshold_Robustness)
Table_G4_Reversal_Threshold_Robustness.to_csv(
    TABLE_DIR / "Table_G4_Reversal_Threshold_Robustness.csv", index=False
)


In [ ]:
# DYNAMIC REVERSAL-THRESHOLD ROBUSTNESS INTERPRETATION
robust = Table_G4_Reversal_Threshold_Robustness.copy()
valid_thresholds = robust.loc[
    robust["ThresholdWithinZeroToBase"], "MRP_UtilityIndifference"
]
if len(valid_thresholds):
    print(
        f"Reversal-threshold sensitivity: across the tested combinations where the indifference "
        f"point lies between zero and the base MRP, the threshold ranges from "
        f"{valid_thresholds.min():.2%} to {valid_thresholds.max():.2%}. "
        "The table also reports the preferred strategy at the base MRP for each risk-aversion "
        "and covariance specification, making clear that the 5.30% base-case threshold is conditional "
        "rather than a universal break point."
    )


### Historical-bootstrap funding diagnostic

The arithmetic expected return and the required compound return are not directly comparable as a probability statement. To add a funding-credibility diagnostic without imposing a parametric return distribution, the notebook resamples complete historical monthly return vectors from the 2015–2022 estimation sample. Each simulated path applies the frozen annual-rebalancing rule for eight years and compares terminal growth-sleeve wealth with the inflation-adjusted 2030 target.

This is a historical-bootstrap sensitivity test. It assumes the empirical monthly return distribution is informative about future outcomes and therefore should not be interpreted as a guaranteed probability of success.


In [ ]:
# ============================================================
# 07D. HISTORICAL-BOOTSTRAP 2030 GOAL DIAGNOSTIC
# ============================================================

BOOTSTRAP_PATHS = 20_000
BOOTSTRAP_MONTHS = 8 * 12
BOOTSTRAP_SEED = 20261017

historical_vectors = returns_for_model[CANDIDATE_TICKERS].to_numpy(dtype=float)
rng = np.random.default_rng(BOOTSTRAP_SEED)

terminal_wealth = np.empty(BOOTSTRAP_PATHS, dtype=float)

for path in range(BOOTSTRAP_PATHS):
    draw_idx = rng.integers(0, len(historical_vectors), size=BOOTSTRAP_MONTHS)
    path_returns = historical_vectors[draw_idx]

    wealth = GROWTH_INVESTED_NOTIONAL
    weights = initial_growth_weights.copy()

    for month_idx, asset_returns in enumerate(path_returns, start=1):
        gross_return = float(weights @ asset_returns)
        wealth *= (1 + gross_return)

        end_values = weights * (1 + asset_returns)
        weights = end_values / end_values.sum()

        # End-December annual rebalance after months 12, 24, ..., 84.
        # The terminal month is not rebalanced because no subsequent return is earned.
        if month_idx % 12 == 0 and month_idx < BOOTSTRAP_MONTHS:
            traded_fraction = float(np.abs(initial_growth_weights - weights).sum())
            wealth *= (1 - BASE_TRADING_COST * traded_fraction)
            weights = initial_growth_weights.copy()

    terminal_wealth[path] = wealth

probability_meeting_target = float(np.mean(terminal_wealth >= NOMINAL_TARGET_2030))
terminal_percentiles = np.percentile(terminal_wealth, [5, 25, 50, 75, 95])

Table_G5_Historical_Bootstrap_Goal = pd.DataFrame({
    "Metric": [
        "Bootstrap paths",
        "Horizon months",
        "Nominal 2030 target",
        "Estimated share of paths meeting target",
        "5th percentile terminal wealth",
        "25th percentile terminal wealth",
        "Median terminal wealth",
        "75th percentile terminal wealth",
        "95th percentile terminal wealth",
    ],
    "Value": [
        BOOTSTRAP_PATHS,
        BOOTSTRAP_MONTHS,
        NOMINAL_TARGET_2030,
        probability_meeting_target,
        *terminal_percentiles,
    ],
})

display(Table_G5_Historical_Bootstrap_Goal)
Table_G5_Historical_Bootstrap_Goal.to_csv(
    TABLE_DIR / "Table_G5_Historical_Bootstrap_Goal.csv", index=False
)

print(
    f"Historical-bootstrap diagnostic: {probability_meeting_target:.1%} of simulated "
    f"eight-year paths meet or exceed the nominal 2030 target of USD {NOMINAL_TARGET_2030:,.0f}. "
    "This is conditional on resampling the historical 2015–2022 monthly return distribution "
    "and is not a guaranteed success probability."
)


In [ ]:
# DYNAMIC 2030 FEASIBILITY INTERPRETATION
margin = recommended_mu - required_growth_cagr
if margin > 0:
    print(
        f"2030 planning result: the required compound return is {required_growth_cagr:.2%}, "
        f"while the recommended portfolio's arithmetic expected return is {recommended_mu:.2%}. "
        f"The resulting margin is {margin:.2%}. This supports feasibility in expectation only; "
        "it is not a probability-of-success estimate or a guaranteed funding margin."
    )
else:
    print(
        f"2030 planning warning: the required compound return is {required_growth_cagr:.2%}, "
        f"which exceeds the recommended portfolio's arithmetic expected return of {recommended_mu:.2%}."
    )


# 8. Initial Decision Record

The initial policy is frozen as of 22 October 2026 for later comparison with the revised mandate.

The decision record summarises the candidate universe, estimation window, benchmark, funded-payment plan, initial dollar allocation, key assumptions, risk-aversion setting and evaluation/rebalancing rule. These items define the policy that may later be reassessed but not rewritten using subsequent information.


### Liability-funding logic

The two mandatory payments are separated from the growth objective. The two-year zero-coupon bond is used for the 2024 payment because its maturity date matches the obligation directly. The five-year Delta bond is used for the 2027 payment, and all interim coupons are retained for that obligation under a 0% reinvestment assumption. This is deliberately conservative because the funding plan does not depend on earning an additional return on the coupons.


In [ ]:
# ============================================================
# 08A. INITIAL DECISION RECORD AND MONITORING RULES
# ============================================================
Table_I1_Initial_Decision_Record = pd.DataFrame({
    "DecisionItem": [
        "Case decision date",
        "Candidate universe",
        "Estimation window",
        "Benchmark",
        "2024 payment funding",
        "2027 payment funding",
        "Recommended growth strategy",
        "Growth capital invested",
        "Base inflation assumption",
        "Base risk aversion",
        "Stock / ETF caps",
        "Evaluation window",
        "Rebalancing rule",
        "Re-estimation in evaluation",
        "Initial trading cost",
    ],
    "FrozenPolicy": [
        str(DECISION_DATE.date()),
        ", ".join(CANDIDATE_TICKERS),
        f"Requested Jan-2015 to Dec-2022; effective common returns "f"{common_dates.min():%b-%Y} to {common_dates.max():%b-%Y} "f"({len(common_dates)} months)",
        BENCHMARK,
        f"Beta zero-coupon; initial cost USD {beta_initial_cost:,.0f}",
        f"Delta + ring-fenced coupons at 0% reinvestment; initial cost USD {delta_initial_cost:,.0f}",
        INITIAL_GROWTH_STRATEGY,
        f"USD {GROWTH_INVESTED_NOTIONAL:,.0f}",
        f"{BASE_INFLATION:.2%}",
        f"A={BASE_RISK_AVERSION}; sensitivity {RISK_AVERSION_VALUES}",
        f"Stock {MAX_STOCK_WEIGHT:.0%}; ETF {MAX_ETF_WEIGHT:.0%}",
        "Jan-2023 to Dec-2025",
        "Annual 31-Dec rebalance to frozen target weights after any scheduled liability payment",
        "None",
        f"{BASE_TRADING_COST:.2%} one-way per dollar traded",
    ],
})

Table_I2_Monitoring_and_Change_Triggers = pd.DataFrame({
    "Monitor": [
        "Liability reserve",
        "Inflation / real-value requirement",
        "Expected-return assumption",
        "Risk tolerance",
        "Concentration and mandate compliance",
    ],
    "EvidenceThatWouldChangePolicy": [
        "Any shortfall in contractual cash coverage or failure to ring-fence Delta coupons",
        "A sustained inflation assumption that materially raises the required CAGR beyond the policy's defensible return range",
        "A forecast deterioration beyond the utility indifference threshold in Table_G3",
        "Trustees formally define a loss metric/horizon that the current allocation violates",
        "A holding breaches eligibility, concentration limits, or the no-borrowing/no-short/no-derivatives mandate",
    ],
    "Action": [
        "Restore funding coverage before taking growth risk",
        "Reassess feasible target, growth risk and trade-off explicitly",
        "Re-evaluate the recommended versus rejected allocation using the predeclared rule",
        "Re-optimize under the newly specified risk preference, documenting the mandate change",
        "Rebalance or replace only under the governing policy and document the change",
    ],
})

display(Table_I1_Initial_Decision_Record)
display(Table_I2_Monitoring_and_Change_Triggers)

Table_I1_Initial_Decision_Record.to_csv(TABLE_DIR / "Table_I1_Initial_Decision_Record.csv", index=False)
Table_I2_Monitoring_and_Change_Triggers.to_csv(TABLE_DIR / "Table_I2_Monitoring_and_Change_Triggers.csv", index=False)


# 9. Revised Mandate Response

This section is intentionally left pending until the trustees release the revised mandate.

Once released, the analysis will retain the original case date, information cutoff and frozen initial policy, and will quantify how the new mandate changes the recommendation. No post-release information will be allowed to enter the original 31 December 2022 decision.


In [ ]:
# ============================================================
# 09A. REVISED MANDATE PLACEHOLDER — DO NOT FABRICATE INPUTS
# ============================================================
REVISED_MANDATE_RELEASE = pd.Timestamp("2026-10-23 16:00:00")
REVISED_MANDATE_STATUS = "Not supplied in reviewed files"

Table_R1_Revised_Mandate_Status = pd.DataFrame({
    "Item": [
        "Scheduled release",
        "Status in this notebook revision",
        "Required response",
        "Information cutoff for recalculation",
        "Original policy treatment",
    ],
    "Value": [
        str(REVISED_MANDATE_RELEASE),
        REVISED_MANDATE_STATUS,
        "Quantify the mandate change after the official revised mandate is provided",
        "31-Dec-2022",
        "Retain unchanged and compare; do not replace retrospectively",
    ],
})

display(Table_R1_Revised_Mandate_Status)
Table_R1_Revised_Mandate_Status.to_csv(TABLE_DIR / "Table_R1_Revised_Mandate_Status.csv", index=False)
